# Training Notebook

To train models (with classification, regression, and joint objectives) on OpenBHB and ADNI embeddings, and evaluate the performance.

In [1]:
import os
import re
import json
import time
import math
import random
import pathlib
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.init import trunc_normal_

import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import pearsonr, ks_2samp, entropy
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split

from mil import AggregateThenClassify
from projector import MLP

In [2]:
@dataclass
class Config:
    # Paths
    openbhb_embeddings_dir: str = "/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings"
    adni_embeddings_dir: str = "/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings"
    openbhb_metadata_tsv: str = "/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv"
    adni_metadata_csv: str = "/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv"

    # Training mode - set to "classification", "regression", "dual"
    task: str = "dual"
    early_stopping: bool = True
    patience: int = 20
    min_delta: float = 1e-4

    # Model/training
    input_dim: int = 768
    hidden_dim: int = 256
    num_classes: int = 3
    batch_size: int = 4
    epochs: int = 100
    learning_rate: float = 1e-3
    weight_decay: float = 0.0
    random_state: int = 42
    test_size: float = 0.2
    attention_type: str = "0"

    # Joint-objective weights
    classification_loss_weight: float = 1.0
    regression_loss_weight: float = 1.0

    # MLP
    mlp_hidden_dims: Optional[List[int]] = None
    norm: Optional[str] = None                   
    act: str = "relu"
    dropout: float = 0.1
    weight_init: str = "kaiming"

    # Saving
    output_root: str = "/home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning"
    run_name: Optional[str] = None # Run names to ID when doing multiple trainings

    # Misc
    save_attention_maps: bool = True
    return_attention_weights: bool = True

LABEL_TO_IDX = {"CN": 0, "MCI": 1, "Dementia": 2}
IDX_TO_LABEL = {v: k for k, v in LABEL_TO_IDX.items()}

In [3]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [4]:
def compute_embedding_stats(folder_path: str) -> Dict[str, float]:
    sum_ = 0.0
    sq_sum = 0.0
    count = 0

    for f in os.listdir(folder_path):
        if f.endswith(".pt"):
            x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()
            sum_ += x.sum().item()
            sq_sum += (x ** 2).sum().item()
            count += x.numel()

    mean = sum_ / count
    std = ((sq_sum / count) - mean ** 2) ** 0.5
    return {"mean": mean, "std": std, "count": count}

In [5]:
def load_combined_dataframe(cfg: Config) -> pd.DataFrame:
    import pathlib
    import re
    import numpy as np
    import pandas as pd
    import torch

    def extract_embedding_tensor(file_path):
        data = torch.load(file_path, map_location="cpu")

        if isinstance(data, torch.Tensor):
            emb_tensor = data

        elif isinstance(data, dict):
            for key in ["embeddings", "embedding", "features", "feats"]:
                if key in data:
                    emb_tensor = data[key]
                    break
            else:
                raise ValueError(
                    f"No embedding tensor key found in {file_path}. "
                    f"Available keys: {list(data.keys())}"
                )
        else:
            raise TypeError(
                f"Unexpected object type in {file_path}: {type(data)}"
            )

        if not isinstance(emb_tensor, torch.Tensor):
            raise TypeError(
                f"Extracted object is not a tensor in {file_path}: {type(emb_tensor)}"
            )

        if emb_tensor.dim() != 2:
            raise ValueError(
                f"Expected embedding shape [N, D] in {file_path}, got {tuple(emb_tensor.shape)}"
            )

        return emb_tensor.detach().cpu()

    path_openbhb = pathlib.Path(cfg.openbhb_embeddings_dir)
    path_adni = pathlib.Path(cfg.adni_embeddings_dir)

    subject_meta = pd.read_csv(cfg.openbhb_metadata_tsv, sep="\t")
    subject_meta["participant_id"] = subject_meta["participant_id"].astype(str)

    adni_meta = pd.read_csv(cfg.adni_metadata_csv)
    adni_meta["subject_id"] = adni_meta["subject_id"].astype(str)
    adni_meta["diagnosis"] = adni_meta["diagnosis"].astype(str)
    adni_meta["age"] = pd.to_numeric(adni_meta["age"], errors="coerce")

    openbhb_pattern = re.compile(r"^sub-([^_]+)_encoder_embeddings.*\.pt$")
    adni_pattern = re.compile(r"^(.+?)_encoder_embeddings.*\.pt$")

    openbhb_data_list = []
    adni_data_list = []

    for file in path_openbhb.glob("*.pt"):
        match = openbhb_pattern.search(file.name)
        if match:
            subj_id = match.group(1)
            emb_tensor = extract_embedding_tensor(file)
            openbhb_data_list.append(
                {
                    "subject id": subj_id,
                    "embedding": emb_tensor,
                    "diagnosis": "CN",
                    "source": "OpenBHB",
                }
            )

    for file in path_adni.glob("*.pt"):
        match = adni_pattern.search(file.name)
        if match:
            subj_id = match.group(1)
            emb_tensor = extract_embedding_tensor(file)
            adni_data_list.append(
                {
                    "subject id": subj_id,
                    "embedding": emb_tensor,
                    "source": "ADNI",
                }
            )

    openbhb_emb_df = pd.DataFrame(openbhb_data_list)
    adni_emb_df = pd.DataFrame(adni_data_list)

    print("\nDataframe sizes before metadata merge:")
    print("OpenBHB embeddings:", openbhb_emb_df.shape)
    print("ADNI embeddings:", adni_emb_df.shape)

    openbhb_df = pd.merge(
        openbhb_emb_df,
        subject_meta,
        left_on="subject id",
        right_on="participant_id",
        how="inner",
        suffixes=("", "_meta"),
    )

    adni_df = pd.merge(
        adni_emb_df,
        adni_meta,
        left_on="subject id",
        right_on="subject_id",
        how="inner",
        suffixes=("", "_meta"),
    )

    print("\nDataframe sizes after metadata merge:")
    print("OpenBHB merged:", openbhb_df.shape)
    print("ADNI merged:", adni_df.shape)

    combined_df = pd.concat([openbhb_df, adni_df], ignore_index=True)

    if "diagnosis" not in combined_df.columns:
        if "diagnosis_x" in combined_df.columns:
            combined_df["diagnosis"] = combined_df["diagnosis_x"]
        elif "diagnosis_y" in combined_df.columns:
            combined_df["diagnosis"] = combined_df["diagnosis_y"]

    combined_df["diagnosis"] = combined_df["diagnosis"].astype(str).str.strip()
    combined_df.loc[
        combined_df["diagnosis"].isin(["None", "nan", "NaN", ""]),
        "diagnosis"
    ] = np.nan

    print("\nDiagnosis distribution before filtering:")
    print(combined_df["diagnosis"].value_counts(dropna=False))

    invalid_diagnoses = combined_df.loc[
        combined_df["diagnosis"].notna() & ~combined_df["diagnosis"].isin(LABEL_TO_IDX),
        "diagnosis"
    ].unique()
    if len(invalid_diagnoses) > 0:
        print("\nUnexpected diagnosis labels found:")
        print(invalid_diagnoses)

    combined_df = combined_df[combined_df["diagnosis"].isin(LABEL_TO_IDX)].copy()

    print("\nDiagnosis distribution after filtering:")
    print(combined_df["diagnosis"].value_counts())

    combined_df["label"] = combined_df["diagnosis"].map(LABEL_TO_IDX)

    combined_df["age"] = pd.to_numeric(combined_df["age"], errors="coerce")

    combined_df = combined_df.dropna(subset=["label", "age"]).copy()
    combined_df["label"] = combined_df["label"].astype(int)
    combined_df["age"] = combined_df["age"].astype(float)
    combined_df["subject id"] = combined_df["subject id"].astype(str)
    combined_df["source"] = combined_df["source"].astype(str)

    print("\nFinal combined dataframe shape:", combined_df.shape)
    print("\nSource distribution:")
    print(combined_df["source"].value_counts())
    print("\nSource × diagnosis breakdown:")
    print(combined_df.groupby("source")["diagnosis"].value_counts())

    return combined_df

In [6]:
def make_train_eval_split(df: pd.DataFrame, cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stratify_col = df["label"] if df["label"].nunique() > 1 else None
    train_df, eval_df = train_test_split(
        df,
        test_size=cfg.test_size,
        random_state=cfg.random_state,
        shuffle=True,
        stratify=stratify_col,
    )
    return train_df.reset_index(drop=True), eval_df.reset_index(drop=True)

In [7]:
class BagDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        self.embeddings = list(self.df["embedding"])
        self.ages = torch.tensor(self.df["age"].values, dtype=torch.float32)
        self.labels = torch.tensor(self.df["label"].values, dtype=torch.long)
        self.subject_ids = self.df["subject id"].astype(str).tolist()
        self.diagnoses = self.df["diagnosis"].astype(str).tolist()

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        return {
            "embedding": self.embeddings[idx],
            "age": self.ages[idx],
            "label": self.labels[idx],
            "subject_id": self.subject_ids[idx],
            "diagnosis": self.diagnoses[idx],
        }

In [8]:
def collate_bags(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    emb_list = [item["embedding"] for item in batch]
    ages = torch.stack([item["age"] for item in batch])
    labels = torch.stack([item["label"] for item in batch])
    subject_ids = [item["subject_id"] for item in batch]
    diagnoses = [item["diagnosis"] for item in batch]

    lengths = [x.shape[0] for x in emb_list]
    cu_seqlens = torch.tensor([0] + np.cumsum(lengths).tolist(), dtype=torch.long)
    emb_cat = torch.cat(emb_list, dim=0)

    return {
        "emb_cat": emb_cat,
        "cu_seqlens": cu_seqlens,
        "ages": ages,
        "labels": labels,
        "subject_ids": subject_ids,
        "diagnoses": diagnoses,
        "lengths": lengths,
    }

In [9]:
def create_dataloaders(train_df: pd.DataFrame, eval_df: pd.DataFrame, cfg: Config):
    train_dataset = BagDataset(train_df)
    eval_dataset = BagDataset(eval_df)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_bags,
    )
    eval_loader = DataLoader(
        eval_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_bags,
    )
    return train_dataset, eval_dataset, train_loader, eval_loader

In [10]:
def compute_class_weights(train_df: pd.DataFrame, num_classes: int) -> torch.Tensor:
    counts = train_df["label"].value_counts().sort_index()
    class_counts = np.array([counts.get(i, 0) for i in range(num_classes)], dtype=np.float64)
    weights = np.zeros(num_classes, dtype=np.float64)
    nonzero = class_counts > 0
    if nonzero.any():
        weights[nonzero] = class_counts.sum() / (num_classes * class_counts[nonzero])
    return torch.tensor(weights, dtype=torch.float32)

In [11]:
class ABMILRegression(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: Optional[int] = None,
        mlp_hidden_dims=None,
        norm=None,
        act="gelu",
        dropout=0.0,
        weight_init="xavier",
        attention_type: str = "0",
    ):
        super().__init__()
        self.pooling = AggregateThenClassify(
            dim=dim,
            hidden_dim=hidden_dim,
            W_out=1,
            attention_type=attention_type,
        )
        self.regression = MLP(
            in_dim=dim,
            out_dim=1,
            hidden_dims=mlp_hidden_dims,
            norm=norm,
            act=act,
            dropout=dropout,
            weight_init=weight_init,
            task="regression",
        )

    def forward(self, emb_list, seq_position, return_attn_weights: bool = False):
        bags, attn_weights = self.pooling(emb_list, seq_position, return_attn_probs=True)
        output = self.regression(bags).squeeze(-1)
        return (output, attn_weights) if return_attn_weights else output


class ABMILClassification(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: Optional[int] = None,
        num_classes: int = 3,
        attention_type: str = "0",
        mlp_hidden_dims=None,
        norm=None,
        act="gelu",
        dropout=0.0,
        weight_init="xavier",
    ):
        super().__init__()
        self.pooling = AggregateThenClassify(
            dim=dim,
            hidden_dim=hidden_dim,
            W_out=1,
            attention_type=attention_type,
        )
        self.classifier = MLP(
            in_dim=dim,
            out_dim=1,  
            hidden_dims=mlp_hidden_dims,
            norm=norm,
            act=act,
            dropout=dropout,
            weight_init=weight_init,
            task="classification",
            num_classes=num_classes,
        )

    def forward(self, emb_list, seq_position, return_attn_weights: bool = False):
        bags, attn_weights = self.pooling(emb_list, seq_position, return_attn_probs=True)
        logits = self.classifier(bags)
        return (logits, attn_weights) if return_attn_weights else logits


class ABMILJack(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: Optional[int] = None,
        num_classes: int = 3,
        attention_type: str = "0",
        mlp_hidden_dims=None,
        norm=None,
        act="gelu",
        dropout=0.0,
        weight_init="xavier",
    ):
        super().__init__()
        self.pooling = AggregateThenClassify(
            dim=dim,
            hidden_dim=hidden_dim,
            W_out=1,
            attention_type=attention_type,
        )
        self.head = MLP(
            in_dim=dim,
            out_dim=1,
            hidden_dims=mlp_hidden_dims,
            norm=norm,
            act=act,
            dropout=dropout,
            weight_init=weight_init,
            task="dual",
            num_classes=num_classes,
        )

    def forward(self, emb_list, seq_position, return_attn_weights: bool = False):
        bags, attn_weights = self.pooling(emb_list, seq_position, return_attn_probs=True)
        age_pred, logits = self.head(bags)

        if age_pred.ndim > 1 and age_pred.shape[-1] == 1:
            age_pred = age_pred.squeeze(-1)

        if return_attn_weights:
            return logits, age_pred, attn_weights
        return logits, age_pred

In [12]:
def extract_subject_attention_maps(attn_batches: List[torch.Tensor], cu_seqlens_batches: List[torch.Tensor]) -> List[torch.Tensor]:
    if not attn_batches:
        return []

    all_attn_weights = torch.cat(attn_batches)
    subject_attention_maps = []
    token_offset = 0

    for batch_cu_seqlens in cu_seqlens_batches:
        for i in range(len(batch_cu_seqlens) - 1):
            start_idx = batch_cu_seqlens[i].item()
            end_idx = batch_cu_seqlens[i + 1].item()
            subject_attn = all_attn_weights[token_offset + start_idx : token_offset + end_idx]
            subject_attention_maps.append(subject_attn.squeeze(-1))
        token_offset += batch_cu_seqlens[-1].item()

    return subject_attention_maps

In [13]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray, bins: int = 20) -> Dict[str, float]:
    mse = np.mean((y_pred - y_true) ** 2)
    mae = np.mean(np.abs(y_pred - y_true))
    rmse = np.sqrt(mse)
    pearson_r, pearson_p = pearsonr(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    residuals = y_pred - y_true
    mean_residual = np.mean(residuals)
    residual_std = np.std(residuals)

    ks_stat, ks_p = ks_2samp(y_true, y_pred)

    hist_range = (min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max()))
    true_hist, bin_edges = np.histogram(y_true, bins=bins, range=hist_range, density=True)
    pred_hist, _ = np.histogram(y_pred, bins=bin_edges, density=True)

    eps = 1e-8
    true_hist = true_hist + eps
    pred_hist = pred_hist + eps
    true_hist = true_hist / true_hist.sum()
    pred_hist = pred_hist / pred_hist.sum()

    kl_true_pred = entropy(true_hist, pred_hist)
    kl_pred_true = entropy(pred_hist, true_hist)
    js_div = 0.5 * entropy(true_hist, 0.5 * (true_hist + pred_hist)) + 0.5 * entropy(pred_hist, 0.5 * (true_hist + pred_hist))

    return {
        "MAE": float(mae),
        "MSE": float(mse),
        "RMSE": float(rmse),
        "R2": float(r2),
        "Pearson_r": float(pearson_r),
        "Pearson_pval": float(pearson_p),
        "Mean_residual": float(mean_residual),
        "Residual_std": float(residual_std),
        "KS_statistic": float(ks_stat),
        "KS_pval": float(ks_p),
        "KL_true_pred": float(kl_true_pred),
        "KL_pred_true": float(kl_pred_true),
        "JS": float(js_div),
        "True_mean": float(np.mean(y_true)),
        "Pred_mean": float(np.mean(y_pred)),
        "True_std": float(np.std(y_true)),
        "Pred_std": float(np.std(y_pred)),
        "True_min": float(np.min(y_true)),
        "Pred_min": float(np.min(y_pred)),
        "True_max": float(np.max(y_true)),
        "Pred_max": float(np.max(y_pred)),
    }



def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),

        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),

        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),

        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

In [14]:
def save_line_plot(x, ys, labels, title, xlabel, ylabel, save_path):
    plt.figure(figsize=(10, 6))
    for y, label in zip(ys, labels):
        plt.plot(x, y, label=label, linewidth=2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()



def save_regression_plots(y_true, y_pred, history, output_dir, prefix="regression", bins=20):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    save_line_plot(
        epochs,
        [history["train_loss"], history["val_loss"]],
        ["Train Loss", "Validation Loss"],
        "Training vs Validation Loss",
        "Epoch",
        "Loss",
        os.path.join(output_dir, f"{prefix}_loss_curve.png"),
    )

    if "train_mae" in history and "val_mae" in history:
        save_line_plot(
            epochs,
            [history["train_mae"], history["val_mae"]],
            ["Train MAE", "Validation MAE"],
            "Training vs Validation MAE",
            "Epoch",
            "MAE",
            os.path.join(output_dir, f"{prefix}_mae_curve.png"),
        )

    plt.figure(figsize=(7, 7))
    plt.scatter(y_true, y_pred, alpha=0.65, s=45)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", linewidth=2, label="Reference: y = x")
    plt.xlabel("True Age")
    plt.ylabel("Predicted Age")
    plt.title("Predicted vs True Age")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{prefix}_scatter_true_vs_pred.png"), dpi=300, bbox_inches="tight")
    plt.close()

    residuals = y_pred - y_true
    residual_std = residuals.std() + 1e-8
    standardized_residuals = (residuals - residuals.mean()) / residual_std

    plt.figure(figsize=(8, 5))
    plt.scatter(y_true, standardized_residuals, alpha=0.65, s=45)
    plt.axhline(0, linestyle="--", linewidth=2)
    plt.axhline(1, linestyle=":", linewidth=1.5, alpha=0.8)
    plt.axhline(-1, linestyle=":", linewidth=1.5, alpha=0.8)
    plt.xlabel("True Age")
    plt.ylabel("Standardized Residual")
    plt.title("Standardized Residuals vs True Age")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{prefix}_residuals_vs_true.png"), dpi=300, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.hist(standardized_residuals, bins=bins, alpha=0.75, edgecolor="black")
    plt.axvline(0, linestyle="--", linewidth=2)
    plt.axvline(1, linestyle=":", linewidth=1.5, alpha=0.8)
    plt.axvline(-1, linestyle=":", linewidth=1.5, alpha=0.8)
    plt.xlabel("Standardized Residual")
    plt.ylabel("Count")
    plt.title("Standardized Residual Distribution")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{prefix}_residual_distribution.png"), dpi=300, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(6, 6))
    stats.probplot(standardized_residuals, dist="norm", plot=plt)
    plt.title("QQ Plot of Standardized Residuals")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{prefix}_qqplot.png"), dpi=300, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.hist(y_true, bins=bins, alpha=0.5, label="True Age", edgecolor="black")
    plt.hist(y_pred, bins=bins, alpha=0.5, label="Predicted Age", edgecolor="black")
    plt.xlabel("Age")
    plt.ylabel("Count")
    plt.title("Distribution of True vs Predicted Ages")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{prefix}_distribution_true_vs_pred.png"), dpi=300, bbox_inches="tight")
    plt.close()


def save_classification_plots(y_true, y_pred, history, output_dir, prefix="classification"):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    save_line_plot(
        epochs,
        [history["train_loss"], history["val_loss"]],
        ["Train Loss", "Validation Loss"],
        "Training vs Validation Cross-Entropy Loss",
        "Epoch",
        "Loss",
        os.path.join(output_dir, f"{prefix}_loss_curve.png"),
    )
    save_line_plot(
        epochs,
        [history["train_accuracy"], history["val_accuracy"]],
        ["Train Accuracy", "Validation Accuracy"],
        "Training vs Validation Accuracy",
        "Epoch",
        "Accuracy (%)",
        os.path.join(output_dir, f"{prefix}_accuracy_curve.png"),
    )

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(LABEL_TO_IDX))))
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title("Confusion Matrix")
    plt.colorbar()
    tick_marks = np.arange(len(LABEL_TO_IDX))
    plt.xticks(tick_marks, list(LABEL_TO_IDX.keys()), rotation=45)
    plt.yticks(tick_marks, list(LABEL_TO_IDX.keys()))
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], "d"), ha="center", va="center")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{prefix}_confusion_matrix.png"), dpi=300, bbox_inches="tight")
    plt.close()

    class_counts = np.bincount(y_true, minlength=len(LABEL_TO_IDX))
    plt.figure(figsize=(7, 5))
    plt.bar(list(LABEL_TO_IDX.keys()), class_counts, alpha=0.8)
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.title("Class Distribution in Evaluation Set")
    plt.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{prefix}_class_distribution.png"), dpi=300, bbox_inches="tight")
    plt.close()

    class_accuracies = []
    for i in range(len(LABEL_TO_IDX)):
        mask = y_true == i
        class_accuracies.append(float(np.mean(y_pred[mask] == i)) if np.sum(mask) > 0 else 0.0)
    plt.figure(figsize=(7, 5))
    plt.bar(list(LABEL_TO_IDX.keys()), class_accuracies, alpha=0.8)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title("Per-Class Accuracy")
    plt.ylim(0, 1)
    plt.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{prefix}_per_class_accuracy.png"), dpi=300, bbox_inches="tight")
    plt.close()



def save_attention_plot(attn_weights: Optional[List[torch.Tensor]], output_dir: str, prefix: str):
    if not attn_weights:
        return
    all_attn = torch.cat(attn_weights).cpu().numpy()
    plt.figure(figsize=(8, 5))
    plt.hist(all_attn, bins=50, edgecolor="black", alpha=0.75)
    plt.xlabel("Attention Weight")
    plt.ylabel("Frequency")
    plt.title("Distribution of Attention Weights")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{prefix}_attention_distribution.png"), dpi=300, bbox_inches="tight")
    plt.close()

In [15]:
class EarlyStopping:
    def __init__(self, patience: int = 10, min_delta: float = 1e-4, mode: str = "min"):
        """
        mode = 'min' for metrics like val_loss / val_mae
        mode = 'max' for metrics like val_accuracy
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode

        self.best_score = None
        self.counter = 0
        self.should_stop = False
        self.best_state_dict = None
        self.best_epoch = None

    def _is_improvement(self, score: float) -> bool:
        if self.best_score is None:
            return True

        if self.mode == "min":
            return score < self.best_score - self.min_delta
        elif self.mode == "max":
            return score > self.best_score + self.min_delta
        else:
            raise ValueError(f"Unsupported mode: {self.mode}")

    def step(self, score: float, model: nn.Module, epoch: int) -> bool:
        """
        Returns True if this epoch improved the monitored metric.
        """
        if self._is_improvement(score):
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
            self.best_state_dict = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            return True

        self.counter += 1
        if self.counter >= self.patience:
            self.should_stop = True
        return False

    def restore_best_weights(self, model: nn.Module, device: torch.device):
        if self.best_state_dict is not None:
            model.load_state_dict(self.best_state_dict)
            model.to(device)

In [16]:
def train_regression(model, train_loader, eval_loader, optimizer, criterion, device, epochs=100, return_attn_weights=False, early_stopper=None):
    history = {"train_loss": [], "val_loss": [], "train_mae": [], "val_mae": []}

    final_eval_true = []
    final_eval_pred = []
    final_attn_weights = []
    final_cu_seqlens = []
    final_subject_ids = []

    for epoch in range(epochs):
        model.train()
        train_loss_total = 0.0
        train_true = []
        train_pred = []

        for batch in train_loader:
            emb_cat = batch["emb_cat"].to(device, dtype=torch.float32)
            cu_seqlens = batch["cu_seqlens"].to(device, dtype=torch.long)
            ages = batch["ages"].to(device, dtype=torch.float32)

            optimizer.zero_grad()
            outputs = model(emb_cat, cu_seqlens)
            loss = criterion(outputs, ages)
            loss.backward()
            optimizer.step()

            train_loss_total += loss.item()
            train_true.append(ages.detach().cpu())
            train_pred.append(outputs.detach().cpu())

        train_true_np = torch.cat(train_true).numpy()
        train_pred_np = torch.cat(train_pred).numpy()
        train_mae = mean_absolute_error(train_true_np, train_pred_np)

        history["train_loss"].append(train_loss_total / len(train_loader))
        history["train_mae"].append(float(train_mae))

        model.eval()
        eval_loss_total = 0.0
        eval_true = []
        eval_pred = []
        eval_attn_batches = []
        eval_cu_seqlens_batches = []
        eval_subject_ids_local = []

        with torch.no_grad():
            for batch in eval_loader:
                emb_cat = batch["emb_cat"].to(device, dtype=torch.float32)
                cu_seqlens = batch["cu_seqlens"].to(device, dtype=torch.long)
                ages = batch["ages"].to(device, dtype=torch.float32)

                if return_attn_weights:
                    outputs, attn_weights = model(emb_cat, cu_seqlens, return_attn_weights=True)
                    eval_attn_batches.append(attn_weights.cpu())
                    eval_cu_seqlens_batches.append(batch["cu_seqlens"].cpu())
                    eval_subject_ids_local.extend(batch["subject_ids"])
                else:
                    outputs = model(emb_cat, cu_seqlens)

                loss = criterion(outputs, ages)
                eval_loss_total += loss.item()
                eval_true.append(ages.cpu())
                eval_pred.append(outputs.cpu())

        eval_true_np = torch.cat(eval_true).numpy()
        eval_pred_np = torch.cat(eval_pred).numpy()
        eval_mae = mean_absolute_error(eval_true_np, eval_pred_np)

        history["val_loss"].append(eval_loss_total / len(eval_loader))
        history["val_mae"].append(float(eval_mae))
        
        if early_stopper is not None:
            early_stopper.step(history["val_loss"][-1], model, epoch + 1)
            if early_stopper.should_stop:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch [{epoch + 1}/{epochs}] | "
                f"Train Loss: {history['train_loss'][-1]:.4f} | "
                f"Val Loss: {history['val_loss'][-1]:.4f} | "
                f"Train MAE: {history['train_mae'][-1]:.4f} | "
                f"Val MAE: {history['val_mae'][-1]:.4f}"
            )

        final_eval_true = eval_true
        final_eval_pred = eval_pred
        final_attn_weights = eval_attn_batches
        final_cu_seqlens = eval_cu_seqlens_batches
        final_subject_ids = eval_subject_ids_local

    if early_stopper is not None:
        early_stopper.restore_best_weights(model, device)
        print(
            f"Restored best regression model from epoch {early_stopper.best_epoch} "
            f"with best val_loss = {early_stopper.best_score:.6f}"
        )

    model.eval()
    final_eval_true = []
    final_eval_pred = []
    final_attn_weights = []
    final_cu_seqlens = []
    final_subject_ids = []
    
    with torch.no_grad():
        for batch in eval_loader:
            emb_cat = batch["emb_cat"].to(device, dtype=torch.float32)
            cu_seqlens = batch["cu_seqlens"].to(device, dtype=torch.long)
            ages = batch["ages"].to(device, dtype=torch.float32)
    
            if return_attn_weights:
                outputs, attn_weights = model(emb_cat, cu_seqlens, return_attn_weights=True)
                final_attn_weights.append(attn_weights.cpu())
                final_cu_seqlens.append(batch["cu_seqlens"].cpu())
                final_subject_ids.extend(batch["subject_ids"])
            else:
                outputs = model(emb_cat, cu_seqlens)
    
            final_eval_true.append(ages.cpu())
            final_eval_pred.append(outputs.cpu())
            
    subject_attention_maps = extract_subject_attention_maps(final_attn_weights, final_cu_seqlens) if return_attn_weights else []
    return {
        "history": history,
        "y_true": torch.cat(final_eval_true).numpy(),
        "y_pred": torch.cat(final_eval_pred).numpy(),
        "attn_weights": subject_attention_maps,
        "subject_ids": final_subject_ids,
    }

In [17]:
def train_classification(model, train_loader, eval_loader, optimizer, criterion, device, epochs=100, return_attn_weights=False, early_stopper=None):
    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}

    final_eval_labels = []
    final_eval_logits = []
    final_attn_weights = []
    final_cu_seqlens = []
    final_subject_ids = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0

        for batch in train_loader:
            emb_cat = batch["emb_cat"].to(device, dtype=torch.float32)
            cu_seqlens = batch["cu_seqlens"].to(device, dtype=torch.long)
            labels = batch["labels"].to(device, dtype=torch.long)

            optimizer.zero_grad()
            logits = model(emb_cat, cu_seqlens)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        history["train_loss"].append(epoch_loss / len(train_loader))
        history["train_accuracy"].append(100.0 * correct / total)

        model.eval()
        eval_loss = 0.0
        correct = 0
        total = 0
        eval_labels = []
        eval_logits = []
        eval_attn_batches = []
        eval_cu_seqlens_batches = []
        eval_subject_ids_local = []

        with torch.no_grad():
            for batch in eval_loader:
                emb_cat = batch["emb_cat"].to(device, dtype=torch.float32)
                cu_seqlens = batch["cu_seqlens"].to(device, dtype=torch.long)
                labels = batch["labels"].to(device, dtype=torch.long)

                if return_attn_weights:
                    logits, attn_weights = model(emb_cat, cu_seqlens, return_attn_weights=True)
                    eval_attn_batches.append(attn_weights.cpu())
                    eval_cu_seqlens_batches.append(batch["cu_seqlens"].cpu())
                    eval_subject_ids_local.extend(batch["subject_ids"])
                else:
                    logits = model(emb_cat, cu_seqlens)

                loss = criterion(logits, labels)
                eval_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()
                eval_labels.append(labels.cpu())
                eval_logits.append(logits.cpu())

        history["val_loss"].append(eval_loss / len(eval_loader))
        history["val_accuracy"].append(100.0 * correct / total)
        
        if early_stopper is not None:
            early_stopper.step(history["val_loss"][-1], model, epoch + 1)
            if early_stopper.should_stop:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch [{epoch + 1}/{epochs}] | "
                f"Train Loss: {history['train_loss'][-1]:.4f}, Acc: {history['train_accuracy'][-1]:.2f}% | "
                f"Val Loss: {history['val_loss'][-1]:.4f}, Acc: {history['val_accuracy'][-1]:.2f}%"
            )

        final_eval_labels = eval_labels
        final_eval_logits = eval_logits
        final_attn_weights = eval_attn_batches
        final_cu_seqlens = eval_cu_seqlens_batches
        final_subject_ids = eval_subject_ids_local

    if early_stopper is not None:
        early_stopper.restore_best_weights(model, device)
        print(
            f"Restored best classification model from epoch {early_stopper.best_epoch} "
            f"with best val_loss = {early_stopper.best_score:.6f}"
        )

    model.eval()

    final_eval_labels = []
    final_eval_logits = []
    final_attn_weights = []
    final_cu_seqlens = []
    final_subject_ids = []
    
    with torch.no_grad():
        for batch in eval_loader:
            emb_cat = batch["emb_cat"].to(device, dtype=torch.float32)
            cu_seqlens = batch["cu_seqlens"].to(device, dtype=torch.long)
            labels = batch["labels"].to(device, dtype=torch.long)
    
            if return_attn_weights:
                logits, attn_weights = model(emb_cat, cu_seqlens, return_attn_weights=True)
                final_attn_weights.append(attn_weights.cpu())
                final_cu_seqlens.append(batch["cu_seqlens"].cpu())
                final_subject_ids.extend(batch["subject_ids"])
            else:
                logits = model(emb_cat, cu_seqlens)
    
            final_eval_labels.append(labels.cpu())
            final_eval_logits.append(logits.cpu())
            
    subject_attention_maps = extract_subject_attention_maps(final_attn_weights, final_cu_seqlens) if return_attn_weights else []
    return {
        "history": history,
        "y_true": torch.cat(final_eval_labels).numpy(),
        "logits": torch.cat(final_eval_logits).numpy(),
        "attn_weights": subject_attention_maps,
        "subject_ids": final_subject_ids,
    }

In [18]:
def train_dual(model, train_loader, eval_loader, optimizer, cls_criterion, reg_criterion, device, epochs=100, cls_weight=1.0, reg_weight=1.0, return_attn_weights=False, early_stopper=None):
    history = {
        "train_total_loss": [],
        "train_cls_loss": [],
        "train_reg_loss": [],
        "train_accuracy": [],
        "train_mae": [],
        "val_total_loss": [],
        "val_cls_loss": [],
        "val_reg_loss": [],
        "val_accuracy": [],
        "val_mae": [],
        # aliases for plotting convenience
        "train_loss": [],
        "val_loss": [],
    }

    final_eval_labels = []
    final_eval_logits = []
    final_eval_ages = []
    final_eval_age_preds = []
    final_attn_weights = []
    final_cu_seqlens = []
    final_subject_ids = []

    for epoch in range(epochs):
        model.train()
        train_total_loss = 0.0
        train_cls_loss = 0.0
        train_reg_loss = 0.0
        train_correct = 0
        train_total = 0
        train_true_ages = []
        train_pred_ages = []

        for batch in train_loader:
            emb_cat = batch["emb_cat"].to(device, dtype=torch.float32)
            cu_seqlens = batch["cu_seqlens"].to(device, dtype=torch.long)
            labels = batch["labels"].to(device, dtype=torch.long)
            ages = batch["ages"].to(device, dtype=torch.float32)

            optimizer.zero_grad()
            logits, age_pred = model(emb_cat, cu_seqlens)

            loss_cls = cls_criterion(logits, labels)
            loss_reg = reg_criterion(age_pred, ages)
            loss = cls_weight * loss_cls + reg_weight * loss_reg

            loss.backward()
            optimizer.step()

            train_total_loss += loss.item()
            train_cls_loss += loss_cls.item()
            train_reg_loss += loss_reg.item()

            preds = torch.argmax(logits, dim=1)
            train_total += labels.size(0)
            train_correct += (preds == labels).sum().item()
            train_true_ages.append(ages.detach().cpu())
            train_pred_ages.append(age_pred.detach().cpu())

        train_true_age_np = torch.cat(train_true_ages).numpy()
        train_pred_age_np = torch.cat(train_pred_ages).numpy()

        history["train_total_loss"].append(train_total_loss / len(train_loader))
        history["train_cls_loss"].append(train_cls_loss / len(train_loader))
        history["train_reg_loss"].append(train_reg_loss / len(train_loader))
        history["train_accuracy"].append(100.0 * train_correct / train_total)
        history["train_mae"].append(float(mean_absolute_error(train_true_age_np, train_pred_age_np)))

        model.eval()
        val_total_loss = 0.0
        val_cls_loss = 0.0
        val_reg_loss = 0.0
        val_correct = 0
        val_total = 0
        eval_labels = []
        eval_logits = []
        eval_ages = []
        eval_age_preds = []
        eval_attn_batches = []
        eval_cu_seqlens_batches = []
        eval_subject_ids_local = []

        with torch.no_grad():
            for batch in eval_loader:
                emb_cat = batch["emb_cat"].to(device, dtype=torch.float32)
                cu_seqlens = batch["cu_seqlens"].to(device, dtype=torch.long)
                labels = batch["labels"].to(device, dtype=torch.long)
                ages = batch["ages"].to(device, dtype=torch.float32)

                if return_attn_weights:
                    logits, age_pred, attn_weights = model(emb_cat, cu_seqlens, return_attn_weights=True)
                    eval_attn_batches.append(attn_weights.cpu())
                    eval_cu_seqlens_batches.append(batch["cu_seqlens"].cpu())
                    eval_subject_ids_local.extend(batch["subject_ids"])
                else:
                    logits, age_pred = model(emb_cat, cu_seqlens)

                loss_cls = cls_criterion(logits, labels)
                loss_reg = reg_criterion(age_pred, ages)
                loss = cls_weight * loss_cls + reg_weight * loss_reg

                val_total_loss += loss.item()
                val_cls_loss += loss_cls.item()
                val_reg_loss += loss_reg.item()

                preds = torch.argmax(logits, dim=1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

                eval_labels.append(labels.cpu())
                eval_logits.append(logits.cpu())
                eval_ages.append(ages.cpu())
                eval_age_preds.append(age_pred.cpu())

        eval_true_age_np = torch.cat(eval_ages).numpy()
        eval_pred_age_np = torch.cat(eval_age_preds).numpy()

        history["val_total_loss"].append(val_total_loss / len(eval_loader))
        history["val_cls_loss"].append(val_cls_loss / len(eval_loader))
        history["val_reg_loss"].append(val_reg_loss / len(eval_loader))
        history["val_accuracy"].append(100.0 * val_correct / val_total)
        history["val_mae"].append(float(mean_absolute_error(eval_true_age_np, eval_pred_age_np)))
        history["train_loss"].append(history["train_total_loss"][-1])
        history["val_loss"].append(history["val_total_loss"][-1])
        
        if early_stopper is not None:
            early_stopper.step(history["val_total_loss"][-1], model, epoch + 1)
            if early_stopper.should_stop:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch [{epoch + 1}/{epochs}] | "
                f"Train Total: {history['train_total_loss'][-1]:.4f} | "
                f"Val Total: {history['val_total_loss'][-1]:.4f} | "
                f"Train Acc: {history['train_accuracy'][-1]:.2f}% | "
                f"Val Acc: {history['val_accuracy'][-1]:.2f}% | "
                f"Train MAE: {history['train_mae'][-1]:.4f} | "
                f"Val MAE: {history['val_mae'][-1]:.4f}"
            )

        final_eval_labels = eval_labels
        final_eval_logits = eval_logits
        final_eval_ages = eval_ages
        final_eval_age_preds = eval_age_preds
        final_attn_weights = eval_attn_batches
        final_cu_seqlens = eval_cu_seqlens_batches
        final_subject_ids = eval_subject_ids_local

    if early_stopper is not None:
        early_stopper.restore_best_weights(model, device)
        print(
            f"Restored best dual model from epoch {early_stopper.best_epoch} "
            f"with best val_total_loss = {early_stopper.best_score:.6f}"
        )

    model.eval()
    
    final_eval_labels = []
    final_eval_logits = []
    final_eval_ages = []
    final_eval_age_preds = []
    final_attn_weights = []
    final_cu_seqlens = []
    final_subject_ids = []
    
    with torch.no_grad():
        for batch in eval_loader:
            emb_cat = batch["emb_cat"].to(device, dtype=torch.float32)
            cu_seqlens = batch["cu_seqlens"].to(device, dtype=torch.long)
            labels = batch["labels"].to(device, dtype=torch.long)
            ages = batch["ages"].to(device, dtype=torch.float32)
    
            if return_attn_weights:
                logits, age_pred, attn_weights = model(emb_cat, cu_seqlens, return_attn_weights=True)
                final_attn_weights.append(attn_weights.cpu())
                final_cu_seqlens.append(batch["cu_seqlens"].cpu())
                final_subject_ids.extend(batch["subject_ids"])
            else:
                logits, age_pred = model(emb_cat, cu_seqlens)
    
            final_eval_labels.append(labels.cpu())
            final_eval_logits.append(logits.cpu())
            final_eval_ages.append(ages.cpu())
            final_eval_age_preds.append(age_pred.cpu())
            
    subject_attention_maps = extract_subject_attention_maps(final_attn_weights, final_cu_seqlens) if return_attn_weights else []
    return {
        "history": history,
        "y_true_cls": torch.cat(final_eval_labels).numpy(),
        "logits": torch.cat(final_eval_logits).numpy(),
        "y_true_reg": torch.cat(final_eval_ages).numpy(),
        "y_pred_reg": torch.cat(final_eval_age_preds).numpy(),
        "attn_weights": subject_attention_maps,
        "subject_ids": final_subject_ids,
    }

In [19]:
def create_output_dir(cfg: Config) -> str:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = cfg.run_name or f"abmil_{cfg.task}_{timestamp}"
    output_dir = os.path.join(cfg.output_root, run_name)
    os.makedirs(output_dir, exist_ok=True)
    return output_dir



def save_json(data: Dict, save_path: str) -> None:
    def convert(obj):
        if isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return str(obj)

    with open(save_path, "w") as f:
        json.dump(data, f, indent=2, default=convert)



def save_attention_csvs(subject_ids: List[str], attn_weights: List[torch.Tensor], output_dir: str):
    if not attn_weights:
        return

    attn_df = pd.DataFrame({"attention_weight": torch.cat(attn_weights).cpu().numpy()})
    attn_df.to_csv(os.path.join(output_dir, "attention_weights.csv"), index=False)

    attn_maps_dir = os.path.join(output_dir, "attention_maps")
    os.makedirs(attn_maps_dir, exist_ok=True)
    for subject_id, subject_attn in zip(subject_ids, attn_weights):
        subject_df = pd.DataFrame(
            {
                "token_index": range(len(subject_attn)),
                "attention_weight": subject_attn.cpu().numpy(),
            }
        )
        subject_df.to_csv(os.path.join(attn_maps_dir, f"subject_{subject_id}_attention.csv"), index=False)


In [20]:
def run_regression(cfg: Config, train_df: pd.DataFrame, eval_df: pd.DataFrame, train_loader, eval_loader, device, output_dir: str):
    model = ABMILRegression(
        dim=cfg.input_dim,
        hidden_dim=cfg.hidden_dim,
        attention_type=cfg.attention_type,
        mlp_hidden_dims=cfg.mlp_hidden_dims,
        norm=cfg.norm,
        act=cfg.act,
        dropout=cfg.dropout,
        weight_init=cfg.weight_init,
    ).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

    early_stopper = EarlyStopping(
        patience=cfg.patience,
        min_delta=cfg.min_delta,
        mode="min",
    ) if cfg.early_stopping else None

    start_time = time.time()
    
    results = train_regression(
        model, train_loader, eval_loader, optimizer, criterion, device,
        epochs=cfg.epochs,
        return_attn_weights=cfg.return_attention_weights,
        early_stopper=early_stopper,
    )
    
    total_time = time.time() - start_time

    y_true = results["y_true"]
    y_pred = results["y_pred"]
    history = results["history"]
    attn_weights = results["attn_weights"]

    metrics = regression_metrics(y_true, y_pred)

    predictions_df = pd.DataFrame(
        {
            "subject_id": results["subject_ids"],
            "actual_age": y_true,
            "predicted_age": y_pred,
            "residual": y_pred - y_true,
            "absolute_error": np.abs(y_pred - y_true),
        }
    )
    predictions_df.to_csv(os.path.join(output_dir, "predictions_regression.csv"), index=False)

    pd.DataFrame(history).to_csv(os.path.join(output_dir, "training_history_regression.csv"), index=False)
    pd.DataFrame([metrics]).to_csv(os.path.join(output_dir, "metrics_regression.csv"), index=False)

    save_regression_plots(y_true, y_pred, history, output_dir, prefix="regression")
    save_attention_plot(attn_weights, output_dir, prefix="regression")
    if cfg.save_attention_maps:
        save_attention_csvs(results["subject_ids"], attn_weights, output_dir)

    summary = {
        "task": "regression",
        "training_time_seconds": total_time,
        "config": asdict(cfg),
        "metrics": metrics,
        "train_size": len(train_df),
        "eval_size": len(eval_df),
    }
    save_json(summary, os.path.join(output_dir, "summary_regression.json"))
    return summary



def run_classification(cfg: Config, train_df: pd.DataFrame, eval_df: pd.DataFrame, train_loader, eval_loader, device, output_dir: str):
    model = ABMILClassification(
        dim=cfg.input_dim,
        hidden_dim=cfg.hidden_dim,
        num_classes=cfg.num_classes,
        attention_type=cfg.attention_type,
        mlp_hidden_dims=cfg.mlp_hidden_dims,
        norm=cfg.norm,
        act=cfg.act,
        dropout=cfg.dropout,
        weight_init=cfg.weight_init,
    ).to(device)
    class_weights = compute_class_weights(train_df, cfg.num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

    early_stopper = EarlyStopping(
        patience=cfg.patience,
        min_delta=cfg.min_delta,
        mode="min",
    ) if cfg.early_stopping else None
    
    start_time = time.time()
    
    results = train_classification(
        model, train_loader, eval_loader, optimizer, criterion, device,
        epochs=cfg.epochs, return_attn_weights=cfg.return_attention_weights, early_stopper=early_stopper,
    )
    
    total_time = time.time() - start_time

    y_true = results["y_true"]
    logits = results["logits"]
    y_pred = np.argmax(logits, axis=1)
    history = results["history"]
    attn_weights = results["attn_weights"]

    metrics = classification_metrics(y_true, y_pred)
    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(cfg.num_classes)),
        target_names=[IDX_TO_LABEL[i] for i in range(cfg.num_classes)],
        zero_division=0,
        output_dict=True,
    )

    predictions_df = pd.DataFrame(
        {
            "subject_id": results["subject_ids"],
            "true_label_idx": y_true,
            "predicted_label_idx": y_pred,
            "true_label": [IDX_TO_LABEL[i] for i in y_true],
            "predicted_label": [IDX_TO_LABEL[i] for i in y_pred],
            "correct": (y_true == y_pred).astype(int),
        }
    )
    predictions_df.to_csv(os.path.join(output_dir, "predictions_classification.csv"), index=False)

    pd.DataFrame(history).to_csv(os.path.join(output_dir, "training_history_classification.csv"), index=False)
    pd.DataFrame([metrics]).to_csv(os.path.join(output_dir, "metrics_classification.csv"), index=False)
    pd.DataFrame(report).transpose().to_csv(os.path.join(output_dir, "classification_report.csv"))

    save_classification_plots(y_true, y_pred, history, output_dir, prefix="classification")
    save_attention_plot(attn_weights, output_dir, prefix="classification")
    if cfg.save_attention_maps:
        save_attention_csvs(results["subject_ids"], attn_weights, output_dir)

    summary = {
        "task": "classification",
        "training_time_seconds": total_time,
        "config": asdict(cfg),
        "class_weights": class_weights.detach().cpu().tolist(),
        "metrics": metrics,
        "train_size": len(train_df),
        "eval_size": len(eval_df),
        "class_counts_train": train_df["diagnosis"].value_counts().to_dict(),
        "class_counts_eval": eval_df["diagnosis"].value_counts().to_dict(),
    }
    save_json(summary, os.path.join(output_dir, "summary_classification.json"))
    return summary



def run_dual(cfg: Config, train_df: pd.DataFrame, eval_df: pd.DataFrame, train_loader, eval_loader, device, output_dir: str):
    model = ABMILJack(
        dim=cfg.input_dim,
        hidden_dim=cfg.hidden_dim,
        num_classes=cfg.num_classes,
        attention_type=cfg.attention_type,
        mlp_hidden_dims=cfg.mlp_hidden_dims,
        norm=cfg.norm,
        act=cfg.act,
        dropout=cfg.dropout,
        weight_init=cfg.weight_init,
    ).to(device)
    class_weights = compute_class_weights(train_df, cfg.num_classes).to(device)
    cls_criterion = nn.CrossEntropyLoss(weight=class_weights)
    reg_criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

    early_stopper = EarlyStopping(
        patience=cfg.patience,
        min_delta=cfg.min_delta,
        mode="min",
    ) if cfg.early_stopping else None
    
    start_time = time.time()
    
    results = train_dual(
        model,
        train_loader,
        eval_loader,
        optimizer,
        cls_criterion,
        reg_criterion,
        device,
        epochs=cfg.epochs,
        cls_weight=cfg.classification_loss_weight,
        reg_weight=cfg.regression_loss_weight,
        return_attn_weights=cfg.return_attention_weights,
        early_stopper=early_stopper,
    )
    total_time = time.time() - start_time

    y_true_cls = results["y_true_cls"]
    logits = results["logits"]
    y_pred_cls = np.argmax(logits, axis=1)
    y_true_reg = results["y_true_reg"]
    y_pred_reg = results["y_pred_reg"]
    history = results["history"]
    attn_weights = results["attn_weights"]

    cls_metrics = classification_metrics(y_true_cls, y_pred_cls)
    reg_metrics = regression_metrics(y_true_reg, y_pred_reg)
    report = classification_report(
        y_true_cls,
        y_pred_cls,
        labels=list(range(cfg.num_classes)),
        target_names=[IDX_TO_LABEL[i] for i in range(cfg.num_classes)],
        zero_division=0,
        output_dict=True,
    )

    predictions_df = pd.DataFrame(
        {
            "subject_id": results["subject_ids"],
            "true_label_idx": y_true_cls,
            "predicted_label_idx": y_pred_cls,
            "true_label": [IDX_TO_LABEL[i] for i in y_true_cls],
            "predicted_label": [IDX_TO_LABEL[i] for i in y_pred_cls],
            "actual_age": y_true_reg,
            "predicted_age": y_pred_reg,
            "age_residual": y_pred_reg - y_true_reg,
            "correct_classification": (y_true_cls == y_pred_cls).astype(int),
        }
    )
    predictions_df.to_csv(os.path.join(output_dir, "predictions_dual.csv"), index=False)

    pd.DataFrame(history).to_csv(os.path.join(output_dir, "training_history_dual.csv"), index=False)
    pd.DataFrame([cls_metrics]).to_csv(os.path.join(output_dir, "metrics_dual_classification.csv"), index=False)
    pd.DataFrame([reg_metrics]).to_csv(os.path.join(output_dir, "metrics_dual_regression.csv"), index=False)
    pd.DataFrame(report).transpose().to_csv(os.path.join(output_dir, "classification_report_dual.csv"))

    save_classification_plots(y_true_cls, y_pred_cls, history, output_dir, prefix="dual_classification")
    save_regression_plots(y_true_reg, y_pred_reg, history, output_dir, prefix="dual_regression")

    epochs = np.arange(1, len(history["train_total_loss"]) + 1)
    save_line_plot(
        epochs,
        [history["train_cls_loss"], history["val_cls_loss"], history["train_reg_loss"], history["val_reg_loss"]],
        ["Train Cls Loss", "Val Cls Loss", "Train Reg Loss", "Val Reg Loss"],
        "Dual-Task Component Losses",
        "Epoch",
        "Loss",
        os.path.join(output_dir, "dual_component_losses.png"),
    )

    save_attention_plot(attn_weights, output_dir, prefix="dual")
    if cfg.save_attention_maps:
        save_attention_csvs(results["subject_ids"], attn_weights, output_dir)

    summary = {
        "task": "dual",
        "training_time_seconds": total_time,
        "config": asdict(cfg),
        "class_weights": class_weights.detach().cpu().tolist(),
        "classification_metrics": cls_metrics,
        "regression_metrics": reg_metrics,
        "train_size": len(train_df),
        "eval_size": len(eval_df),
        "class_counts_train": train_df["diagnosis"].value_counts().to_dict(),
        "class_counts_eval": eval_df["diagnosis"].value_counts().to_dict(),
    }
    save_json(summary, os.path.join(output_dir, "summary_dual.json"))
    return summary


In [21]:
def main(cfg: Optional[Config] = None):
    cfg = cfg or Config()
    set_seed(cfg.random_state)

    print("Loading data...")
    combined_df = load_combined_dataframe(cfg)
    train_df, eval_df = make_train_eval_split(combined_df, cfg)
    _, _, train_loader, eval_loader = create_dataloaders(train_df, eval_df, cfg)

    stats_dict = compute_embedding_stats(cfg.openbhb_embeddings_dir)
    print(f"OpenBHB embedding mean: {stats_dict['mean']:.6f}")
    print(f"OpenBHB embedding std:  {stats_dict['std']:.6f}")
    print("Label distribution (full dataset):")
    print(combined_df["diagnosis"].value_counts())

    output_dir = create_output_dir(cfg)
    save_json({"config": asdict(cfg), "embedding_stats": stats_dict}, os.path.join(output_dir, "run_config.json"))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    print(f"Saving outputs to: {output_dir}")

    if cfg.task == "regression":
        summary = run_regression(cfg, train_df, eval_df, train_loader, eval_loader, device, output_dir)
    elif cfg.task == "classification":
        summary = run_classification(cfg, train_df, eval_df, train_loader, eval_loader, device, output_dir)
    elif cfg.task == "dual":
        summary = run_dual(cfg, train_df, eval_df, train_loader, eval_loader, device, output_dir)
    else:
        raise ValueError(f"Unsupported task: {cfg.task}. Choose from 'classification', 'regression', or 'dual'.")

    print("\nRun complete.")
    print(json.dumps(summary, indent=2, default=str))
    return summary

In [21]:
cfg = Config(task = "classification", epochs = 200, batch_size = 16)
main(cfg)

Loading data...


/tmp/ipykernel_80063/3217860002.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emb_tensor = torch.load(file, map_location="cpu")
/tmp/ipykernel_80063/3217860002.py:44:


Raw dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (387, 4)

After metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (387, 79)

Diagnosis distribution BEFORE filtering:
diagnosis
CN          1492
MCI          186
Dementia      39
Name: count, dtype: int64

Diagnosis distribution AFTER filtering:
diagnosis
CN          1492
MCI          186
Dementia      39
Name: count, dtype: int64

Final combined dataframe shape: (1717, 90)

Source distribution:
source
OpenBHB    1330
ADNI        387
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     MCI           186
         CN            162
         Dementia       39
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_80063/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
MCI          186
Dementia      39
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_baseline/abmil_classification_20260419_015450
Epoch [10/200] | Train Loss: 0.2055, Acc: 90.82% | Val Loss: 0.6789, Acc: 86.05%
Epoch [20/200] | Train Loss: 0.0101, Acc: 99.85% | Val Loss: 1.4290, Acc: 88.66%
Epoch [30/200] | Train Loss: 0.0014, Acc: 100.00% | Val Loss: 1.5205, Acc: 88.66%
Epoch [40/200] | Train Loss: 0.0006, Acc: 100.00% | Val Loss: 1.6856, Acc: 87.79%
Epoch [50/200] | Train Loss: 0.0003, Acc: 100.00% | Val Loss: 1.8344, Acc: 87.79%
Epoch [60/200] | Train Loss: 0.0003, Acc: 100.00% | Val Loss: 1.9125, Acc: 87.79%
Epoch [70/200] | Train Loss: 0.0001, Acc: 100.00% | Val Loss: 2.1287, Acc: 88.37%
Epoch [80/200] | Train Loss: 0.0001, Acc: 100.00% | Val Loss: 2.2261, Acc: 88.08%
Epoch [90/200] | Train

{'task': 'classification',
 'training_time_seconds': 354.07044076919556,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_tsv': '/home/varnika/neurovfm/experiments/data/ADNI_train_embeddings/00_adni_metadata.tsv',
  'diagnosis_labels_tsv': '/home/varnika/neurovfm/experiments/data/ADNI_train_embeddings/00_diagonsis_label.tsv',
  'task': 'classification',
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 16,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'output_root': '/home/varnika/neurovfm/experiments/outputs/results_baseline',
  'run_name': Non

In [22]:
cfg = Config(task = "regression", epochs = 200, batch_size = 16)
main(cfg)

Loading data...


/tmp/ipykernel_80063/3217860002.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emb_tensor = torch.load(file, map_location="cpu")
/tmp/ipykernel_80063/3217860002.py:44:


Raw dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (387, 4)

After metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (387, 79)

Diagnosis distribution BEFORE filtering:
diagnosis
CN          1492
MCI          186
Dementia      39
Name: count, dtype: int64

Diagnosis distribution AFTER filtering:
diagnosis
CN          1492
MCI          186
Dementia      39
Name: count, dtype: int64

Final combined dataframe shape: (1717, 90)

Source distribution:
source
OpenBHB    1330
ADNI        387
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     MCI           186
         CN            162
         Dementia       39
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_80063/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
MCI          186
Dementia      39
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_baseline/abmil_regression_20260419_020053
Epoch [10/200] | Train Loss: 30.1251 | Val Loss: 34.9631 | Train MAE: 3.9969 | Val MAE: 4.4512
Epoch [20/200] | Train Loss: 12.0677 | Val Loss: 30.1024 | Train MAE: 2.5542 | Val MAE: 3.8611
Epoch [30/200] | Train Loss: 7.1790 | Val Loss: 27.3062 | Train MAE: 1.9937 | Val MAE: 3.7727
Epoch [40/200] | Train Loss: 5.2158 | Val Loss: 31.6413 | Train MAE: 1.7415 | Val MAE: 3.9585
Epoch [50/200] | Train Loss: 4.6975 | Val Loss: 30.2298 | Train MAE: 1.6184 | Val MAE: 3.8935
Epoch [60/200] | Train Loss: 2.8558 | Val Loss: 31.2755 | Train MAE: 1.3071 | Val MAE: 3.9847
Epoch [70/200] | Train Loss: 1.8266 | Val Loss: 32.1037 | Train MAE: 1.0474 | Val MAE: 4.0248
Epoch [80/200] | Tra

{'task': 'regression',
 'training_time_seconds': 348.55286931991577,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_tsv': '/home/varnika/neurovfm/experiments/data/ADNI_train_embeddings/00_adni_metadata.tsv',
  'diagnosis_labels_tsv': '/home/varnika/neurovfm/experiments/data/ADNI_train_embeddings/00_diagonsis_label.tsv',
  'task': 'regression',
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 16,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'output_root': '/home/varnika/neurovfm/experiments/outputs/results_baseline',
  'run_name': None,
  'sa

In [23]:
cfg = Config(task = "dual", epochs = 200, batch_size = 16)
main(cfg)

Loading data...


/tmp/ipykernel_80063/3217860002.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emb_tensor = torch.load(file, map_location="cpu")
/tmp/ipykernel_80063/3217860002.py:44:


Raw dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (387, 4)

After metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (387, 79)

Diagnosis distribution BEFORE filtering:
diagnosis
CN          1492
MCI          186
Dementia      39
Name: count, dtype: int64

Diagnosis distribution AFTER filtering:
diagnosis
CN          1492
MCI          186
Dementia      39
Name: count, dtype: int64

Final combined dataframe shape: (1717, 90)

Source distribution:
source
OpenBHB    1330
ADNI        387
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     MCI           186
         CN            162
         Dementia       39
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_80063/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
MCI          186
Dementia      39
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_baseline/abmil_dual_20260419_020650
Epoch [10/200] | Train Total: 25.7700 | Val Total: 39.1309 | Train Acc: 90.53% | Val Acc: 86.92% | Train MAE: 3.6472 | Val MAE: 4.5303
Epoch [20/200] | Train Total: 12.4309 | Val Total: 31.2451 | Train Acc: 91.41% | Val Acc: 86.05% | Train MAE: 2.5534 | Val MAE: 4.0855
Epoch [30/200] | Train Total: 7.5727 | Val Total: 35.4207 | Train Acc: 93.95% | Val Acc: 86.92% | Train MAE: 2.0252 | Val MAE: 4.1513
Epoch [40/200] | Train Total: 5.6597 | Val Total: 31.1584 | Train Acc: 94.61% | Val Acc: 87.21% | Train MAE: 1.7380 | Val MAE: 4.0698
Epoch [50/200] | Train Total: 3.7176 | Val Total: 31.8724 | Train Acc: 94.90% | Val Acc: 87.50% | Train MAE: 1.3802 | Val MAE: 4.0738
Epoch [60/200]

{'task': 'dual',
 'training_time_seconds': 356.44077157974243,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_tsv': '/home/varnika/neurovfm/experiments/data/ADNI_train_embeddings/00_adni_metadata.tsv',
  'diagnosis_labels_tsv': '/home/varnika/neurovfm/experiments/data/ADNI_train_embeddings/00_diagonsis_label.tsv',
  'task': 'dual',
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 16,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'output_root': '/home/varnika/neurovfm/experiments/outputs/results_baseline',
  'run_name': None,
  'save_attention

### Runs with best hyperparameters obtained from hyperparameter tuning

In [32]:
cfg = Config(task = "classification", epochs = 200, hidden_dim = 256, batch_size = 8, learning_rate = 0.0003, weight_decay = 0.0, attention_type = "0")
main(cfg)

Loading data...


/tmp/ipykernel_4156593/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4156593/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_classification_20260422_142528
Epoch [10/200] | Train Loss: 0.1273, Acc: 93.78% | Val Loss: 0.5168, Acc: 85.68%
Epoch [20/200] | Train Loss: 0.0381, Acc: 98.99% | Val Loss: 0.8516, Acc: 86.22%
Early stopping triggered at epoch 25
Restored best classification model from epoch 5 with best val_loss = 0.360108

Run complete.
{
  "task": "classification",
  "training_time_seconds": 49.72103142738342,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings",
    "adni_embeddings_dir": "/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings",
    "openbhb_metadata_tsv": "/home/varnika/neurovfm/experiments/data/OpenBHB_tra

{'task': 'classification',
 'training_time_seconds': 49.72103142738342,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'classification',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 8,
  'epochs': 200,
  'learning_rate': 0.0003,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '0',
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output

In [33]:
cfg = Config(task = "classification", epochs = 200, hidden_dim = 256, batch_size = 8, learning_rate = 0.0003, weight_decay = 0.0, attention_type = "1")
main(cfg)

Loading data...


/tmp/ipykernel_4156593/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4156593/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_classification_20260422_142624
Epoch [10/200] | Train Loss: 0.1279, Acc: 93.78% | Val Loss: 0.5560, Acc: 85.41%
Epoch [20/200] | Train Loss: 0.0738, Acc: 97.97% | Val Loss: 1.4671, Acc: 84.32%
Early stopping triggered at epoch 25
Restored best classification model from epoch 5 with best val_loss = 0.377312

Run complete.
{
  "task": "classification",
  "training_time_seconds": 44.853885889053345,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings",
    "adni_embeddings_dir": "/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings",
    "openbhb_metadata_tsv": "/home/varnika/neurovfm/experiments/data/OpenBHB_tr

{'task': 'classification',
 'training_time_seconds': 44.853885889053345,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'classification',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 8,
  'epochs': 200,
  'learning_rate': 0.0003,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '1',
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'outpu

In [22]:
cfg = Config(task = "classification", epochs = 200, hidden_dim = 256, batch_size = 8, learning_rate = 0.0003, weight_decay = 0.0, attention_type = "2")
main(cfg)

Loading data...


/tmp/ipykernel_4185220/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4185220/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_classification_20260422_143323
Epoch [10/200] | Train Loss: 0.2543, Acc: 90.68% | Val Loss: 0.3423, Acc: 86.49%
Epoch [20/200] | Train Loss: 0.1568, Acc: 93.58% | Val Loss: 0.3601, Acc: 86.49%
Epoch [30/200] | Train Loss: 0.0990, Acc: 95.61% | Val Loss: 0.4511, Acc: 87.03%
Early stopping triggered at epoch 32
Restored best classification model from epoch 12 with best val_loss = 0.331792

Run complete.
{
  "task": "classification",
  "training_time_seconds": 51.34153366088867,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings",
    "adni_embeddings_dir": "/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings"

{'task': 'classification',
 'training_time_seconds': 51.34153366088867,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'classification',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 8,
  'epochs': 200,
  'learning_rate': 0.0003,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '2',
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output

In [22]:
cfg = Config(task = "classification", epochs = 200, hidden_dim = 256, batch_size = 8, learning_rate = 0.0003, weight_decay = 0.0, attention_type = "3")
main(cfg)

Loading data...


/tmp/ipykernel_4185735/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4185735/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_classification_20260422_143651
Epoch [10/200] | Train Loss: 0.1023, Acc: 94.05% | Val Loss: 0.4856, Acc: 86.49%
Epoch [20/200] | Train Loss: 0.0674, Acc: 96.69% | Val Loss: 0.5883, Acc: 87.30%
Early stopping triggered at epoch 26
Restored best classification model from epoch 6 with best val_loss = 0.370414

Run complete.
{
  "task": "classification",
  "training_time_seconds": 45.85738492012024,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings",
    "adni_embeddings_dir": "/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings",
    "openbhb_metadata_tsv": "/home/varnika/neurovfm/experiments/data/OpenBHB_tra

{'task': 'classification',
 'training_time_seconds': 45.85738492012024,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'classification',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 8,
  'epochs': 200,
  'learning_rate': 0.0003,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '3',
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output

In [22]:
cfg = Config(task = "classification", epochs = 200, hidden_dim = 256, batch_size = 8, learning_rate = 0.0003, weight_decay = 0.0, attention_type = "5")
main(cfg)

Loading data...


/tmp/ipykernel_4189528/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4189528/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_classification_20260422_144028
Epoch [10/200] | Train Loss: 0.1243, Acc: 94.39% | Val Loss: 0.5954, Acc: 85.68%
Epoch [20/200] | Train Loss: 0.0558, Acc: 97.64% | Val Loss: 0.6609, Acc: 87.57%
Early stopping triggered at epoch 22
Restored best classification model from epoch 2 with best val_loss = 0.411985

Run complete.
{
  "task": "classification",
  "training_time_seconds": 42.95654821395874,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings",
    "adni_embeddings_dir": "/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings",
    "openbhb_metadata_tsv": "/home/varnika/neurovfm/experiments/data/OpenBHB_tra

{'task': 'classification',
 'training_time_seconds': 42.95654821395874,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'classification',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 8,
  'epochs': 200,
  'learning_rate': 0.0003,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '5',
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output

In [23]:
cfg = Config(task = "regression", epochs = 200, hidden_dim = 128, batch_size = 8, learning_rate = 0.001, weight_decay = 0.0, attention_type = "0")
main(cfg)

Loading data...


/tmp/ipykernel_4189528/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4189528/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_regression_20260422_144119
Epoch [10/200] | Train Loss: 22.8735 | Val Loss: 33.5825 | Train MAE: 3.5139 | Val MAE: 4.0397
Epoch [20/200] | Train Loss: 8.5436 | Val Loss: 32.3233 | Train MAE: 2.2305 | Val MAE: 3.9594
Epoch [30/200] | Train Loss: 5.6575 | Val Loss: 32.0203 | Train MAE: 1.8125 | Val MAE: 3.9466
Early stopping triggered at epoch 33
Restored best regression model from epoch 13 with best val_loss = 31.050708

Run complete.
{
  "task": "regression",
  "training_time_seconds": 62.12771487236023,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings",
    "adni_embeddings_dir": "/home/varnika/neurovfm/experiments/da

{'task': 'regression',
 'training_time_seconds': 62.12771487236023,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'regression',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 128,
  'num_classes': 3,
  'batch_size': 8,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '0',
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output_root': '

In [24]:
cfg = Config(task = "regression", epochs = 200, hidden_dim = 128, batch_size = 8, learning_rate = 0.001, weight_decay = 0.0, attention_type = "1")
main(cfg)

Loading data...


/tmp/ipykernel_4189528/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4189528/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_regression_20260422_144229
Epoch [10/200] | Train Loss: 22.4095 | Val Loss: 31.1810 | Train MAE: 3.4429 | Val MAE: 3.9514
Epoch [20/200] | Train Loss: 9.4397 | Val Loss: 30.5835 | Train MAE: 2.3186 | Val MAE: 3.8247
Epoch [30/200] | Train Loss: 5.4969 | Val Loss: 43.4749 | Train MAE: 1.7824 | Val MAE: 4.1886
Early stopping triggered at epoch 40
Restored best regression model from epoch 20 with best val_loss = 30.583509

Run complete.
{
  "task": "regression",
  "training_time_seconds": 69.35925388336182,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings",
    "adni_embeddings_dir": "/home/varnika/neurovfm/experiments/da

{'task': 'regression',
 'training_time_seconds': 69.35925388336182,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'regression',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 128,
  'num_classes': 3,
  'batch_size': 8,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '1',
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output_root': '

In [25]:
cfg = Config(task = "regression", epochs = 200, hidden_dim = 128, batch_size = 8, learning_rate = 0.001, weight_decay = 0.0, attention_type = "2")
main(cfg)

Loading data...


/tmp/ipykernel_4189528/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4189528/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_regression_20260422_144347
Epoch [10/200] | Train Loss: 48.3959 | Val Loss: 47.2962 | Train MAE: 5.0688 | Val MAE: 4.8918
Epoch [20/200] | Train Loss: 33.5377 | Val Loss: 38.0201 | Train MAE: 4.2232 | Val MAE: 4.4501
Epoch [30/200] | Train Loss: 26.6793 | Val Loss: 36.3335 | Train MAE: 3.7632 | Val MAE: 4.5510
Epoch [40/200] | Train Loss: 23.8551 | Val Loss: 38.4053 | Train MAE: 3.5836 | Val MAE: 4.7478
Early stopping triggered at epoch 42
Restored best regression model from epoch 22 with best val_loss = 32.212825

Run complete.
{
  "task": "regression",
  "training_time_seconds": 69.86526536941528,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/experiments/

{'task': 'regression',
 'training_time_seconds': 69.86526536941528,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'regression',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 128,
  'num_classes': 3,
  'batch_size': 8,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '2',
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output_root': '

In [26]:
cfg = Config(task = "regression", epochs = 200, hidden_dim = 128, batch_size = 8, learning_rate = 0.001, weight_decay = 0.0, attention_type = "3")
main(cfg)

Loading data...


/tmp/ipykernel_4189528/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4189528/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_regression_20260422_144504
Epoch [10/200] | Train Loss: 35.0295 | Val Loss: 48.3207 | Train MAE: 4.3604 | Val MAE: 4.9565
Epoch [20/200] | Train Loss: 22.7319 | Val Loss: 43.8337 | Train MAE: 3.5266 | Val MAE: 4.6765
Epoch [30/200] | Train Loss: 10.6748 | Val Loss: 40.5387 | Train MAE: 2.4508 | Val MAE: 4.5985
Epoch [40/200] | Train Loss: 7.9781 | Val Loss: 41.6946 | Train MAE: 2.1467 | Val MAE: 4.5735
Epoch [50/200] | Train Loss: 5.8405 | Val Loss: 39.7837 | Train MAE: 1.8201 | Val MAE: 4.6292
Epoch [60/200] | Train Loss: 5.7892 | Val Loss: 41.6268 | Train MAE: 1.8061 | Val MAE: 4.6394
Early stopping triggered at epoch 70
Restored best regression model from epoch 50 with best v

{'task': 'regression',
 'training_time_seconds': 124.90975069999695,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'regression',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 128,
  'num_classes': 3,
  'batch_size': 8,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '3',
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output_root': 

In [27]:
cfg = Config(task = "regression", epochs = 200, hidden_dim = 128, batch_size = 8, learning_rate = 0.001, weight_decay = 0.0, attention_type = "5")
main(cfg)

Loading data...


/tmp/ipykernel_4189528/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4189528/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_regression_20260422_144717
Epoch [10/200] | Train Loss: 19.6673 | Val Loss: 29.8605 | Train MAE: 3.2519 | Val MAE: 3.9537
Epoch [20/200] | Train Loss: 8.7175 | Val Loss: 34.7931 | Train MAE: 2.2701 | Val MAE: 4.1410
Epoch [30/200] | Train Loss: 4.7618 | Val Loss: 37.2079 | Train MAE: 1.6655 | Val MAE: 4.1216
Early stopping triggered at epoch 35
Restored best regression model from epoch 15 with best val_loss = 28.959425

Run complete.
{
  "task": "regression",
  "training_time_seconds": 65.24846291542053,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings",
    "adni_embeddings_dir": "/home/varnika/neurovfm/experiments/da

{'task': 'regression',
 'training_time_seconds': 65.24846291542053,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'regression',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 128,
  'num_classes': 3,
  'batch_size': 8,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '5',
  'classification_loss_weight': 1.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output_root': '

In [28]:
cfg = Config(task = "dual", epochs = 200, hidden_dim = 256, batch_size = 16, learning_rate = 0.001, classification_loss_weight = 2.0, attention_type = "0")
main(cfg)

Loading data...


/tmp/ipykernel_4189528/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4189528/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_dual_20260422_144831
Epoch [10/200] | Train Total: 22.7764 | Val Total: 29.0733 | Train Acc: 90.47% | Val Acc: 85.95% | Train MAE: 3.5643 | Val MAE: 4.0512
Epoch [20/200] | Train Total: 12.0741 | Val Total: 30.9718 | Train Acc: 91.69% | Val Acc: 86.49% | Train MAE: 2.6122 | Val MAE: 3.8507
Epoch [30/200] | Train Total: 7.2901 | Val Total: 44.8094 | Train Acc: 93.78% | Val Acc: 85.95% | Train MAE: 2.0454 | Val MAE: 4.4560
Early stopping triggered at epoch 33
Restored best dual model from epoch 13 with best val_total_loss = 26.050422

Run complete.
{
  "task": "dual",
  "training_time_seconds": 68.2117714881897,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/e

{'task': 'dual',
 'training_time_seconds': 68.2117714881897,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'dual',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 16,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '0',
  'classification_loss_weight': 2.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output_root': '/home/varnik

In [29]:
cfg = Config(task = "dual", epochs = 200, hidden_dim = 256, batch_size = 16, learning_rate = 0.001, classification_loss_weight = 2.0, attention_type = "1")
main(cfg)

Loading data...


/tmp/ipykernel_4189528/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4189528/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_dual_20260422_144949
Epoch [10/200] | Train Total: 23.8889 | Val Total: 26.1838 | Train Acc: 90.27% | Val Acc: 85.95% | Train MAE: 3.6494 | Val MAE: 3.8597
Epoch [20/200] | Train Total: 14.2909 | Val Total: 29.2346 | Train Acc: 91.69% | Val Acc: 86.49% | Train MAE: 2.8120 | Val MAE: 3.7321
Epoch [30/200] | Train Total: 7.1900 | Val Total: 35.3362 | Train Acc: 93.65% | Val Acc: 85.68% | Train MAE: 2.0689 | Val MAE: 4.0336
Early stopping triggered at epoch 33
Restored best dual model from epoch 13 with best val_total_loss = 25.362297

Run complete.
{
  "task": "dual",
  "training_time_seconds": 63.67971897125244,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/

{'task': 'dual',
 'training_time_seconds': 63.67971897125244,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'dual',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 16,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '1',
  'classification_loss_weight': 2.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output_root': '/home/varni

In [30]:
cfg = Config(task = "dual", epochs = 200, hidden_dim = 256, batch_size = 16, learning_rate = 0.001, classification_loss_weight = 2.0, attention_type = "2")
main(cfg)

Loading data...


/tmp/ipykernel_4189528/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4189528/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_dual_20260422_145104
Epoch [10/200] | Train Total: 26.9243 | Val Total: 29.3565 | Train Acc: 90.47% | Val Acc: 85.68% | Train MAE: 3.7985 | Val MAE: 3.9830
Epoch [20/200] | Train Total: 19.9187 | Val Total: 28.7691 | Train Acc: 91.89% | Val Acc: 86.49% | Train MAE: 3.2641 | Val MAE: 3.7526
Epoch [30/200] | Train Total: 15.1332 | Val Total: 30.4340 | Train Acc: 93.45% | Val Acc: 85.68% | Train MAE: 2.8908 | Val MAE: 3.8343
Early stopping triggered at epoch 38
Restored best dual model from epoch 18 with best val_total_loss = 26.295044

Run complete.
{
  "task": "dual",
  "training_time_seconds": 65.26960229873657,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm

{'task': 'dual',
 'training_time_seconds': 65.26960229873657,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'dual',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 16,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '2',
  'classification_loss_weight': 2.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output_root': '/home/varni

In [31]:
cfg = Config(task = "dual", epochs = 200, hidden_dim = 256, batch_size = 16, learning_rate = 0.001, classification_loss_weight = 2.0, attention_type = "3")
main(cfg)

Loading data...


/tmp/ipykernel_4189528/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4189528/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_dual_20260422_145220
Epoch [10/200] | Train Total: 23.1617 | Val Total: 38.9516 | Train Acc: 90.74% | Val Acc: 85.68% | Train MAE: 3.5951 | Val MAE: 4.7403
Epoch [20/200] | Train Total: 14.6004 | Val Total: 33.7145 | Train Acc: 92.23% | Val Acc: 87.30% | Train MAE: 2.8567 | Val MAE: 4.0103
Epoch [30/200] | Train Total: 8.1370 | Val Total: 40.0423 | Train Acc: 94.73% | Val Acc: 87.30% | Train MAE: 2.1849 | Val MAE: 4.1256
Epoch [40/200] | Train Total: 8.7618 | Val Total: 36.7895 | Train Acc: 95.54% | Val Acc: 88.11% | Train MAE: 2.2763 | Val MAE: 4.0832
Early stopping triggered at epoch 48
Restored best dual model from epoch 28 with best val_total_loss = 29.947756

Run complete.


{'task': 'dual',
 'training_time_seconds': 92.01186728477478,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'dual',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 16,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '3',
  'classification_loss_weight': 2.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output_root': '/home/varni

In [32]:
cfg = Config(task = "dual", epochs = 200, hidden_dim = 256, batch_size = 16, learning_rate = 0.001, classification_loss_weight = 2.0, attention_type = "5")
main(cfg)

Loading data...


/tmp/ipykernel_4189528/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64


/tmp/ipykernel_4189528/2744742967.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load(os.path.join(folder_path, f), map_location="cpu").float()


OpenBHB embedding mean: -0.003446
OpenBHB embedding std:  0.058552
Label distribution (full dataset):
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64
Using device: cuda
Saving outputs to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning/abmil_dual_20260422_145402
Epoch [10/200] | Train Total: 22.7969 | Val Total: 44.4063 | Train Acc: 90.14% | Val Acc: 85.14% | Train MAE: 3.5784 | Val MAE: 5.0359
Epoch [20/200] | Train Total: 12.3841 | Val Total: 37.1383 | Train Acc: 92.09% | Val Acc: 87.03% | Train MAE: 2.6569 | Val MAE: 4.0467
Epoch [30/200] | Train Total: 6.9305 | Val Total: 33.6036 | Train Acc: 94.46% | Val Acc: 86.76% | Train MAE: 2.0338 | Val MAE: 3.8008
Early stopping triggered at epoch 32
Restored best dual model from epoch 12 with best val_total_loss = 27.793188

Run complete.
{
  "task": "dual",
  "training_time_seconds": 66.5278971195221,
  "config": {
    "openbhb_embeddings_dir": "/home/varnika/neurovfm/e

{'task': 'dual',
 'training_time_seconds': 66.5278971195221,
 'config': {'openbhb_embeddings_dir': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings',
  'adni_embeddings_dir': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings',
  'openbhb_metadata_tsv': '/home/varnika/neurovfm/experiments/data/OpenBHB_train_embeddings/for_training_participants.tsv',
  'adni_metadata_csv': '/home/varnika/neurovfm/experiments/data/New_ADNI_train_embeddings/adni_metadata.csv',
  'task': 'dual',
  'early_stopping': True,
  'patience': 20,
  'min_delta': 0.0001,
  'input_dim': 768,
  'hidden_dim': 256,
  'num_classes': 3,
  'batch_size': 16,
  'epochs': 200,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'random_state': 42,
  'test_size': 0.2,
  'attention_type': '5',
  'classification_loss_weight': 2.0,
  'regression_loss_weight': 1.0,
  'mlp_hidden_dims': None,
  'norm': None,
  'act': 'relu',
  'dropout': 0.1,
  'weight_init': 'kaiming',
  'output_root': '/home/varnik

In [33]:
root = "/home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_fn_tuning"
out_txt = os.path.join(root, "all_results_summary.txt")

all_records = []

with open(out_txt, "w") as f:

    for subdir in sorted(os.listdir(root)):
        subpath = os.path.join(root, subdir)
        if not os.path.isdir(subpath):
            continue

        summary_file = None
        for file in os.listdir(subpath):
            if file.startswith("summary") and file.endswith(".json"):
                summary_file = os.path.join(subpath, file)
                break

        if summary_file is None:
            continue

        try:
            with open(summary_file, "r") as sf:
                summary = json.load(sf)
        except:
            continue

        task = summary.get("task", "unknown")

        record = {
            "folder": subdir,
            "task": task,
        }

        f.write("=" * 80 + "\n")
        f.write(f"Folder: {subdir}\n")
        f.write(f"Task: {task}\n")

        if task == "classification":
            m = summary["metrics"]
            record.update({
                "accuracy": m["accuracy"],
                "f1_macro": m["f1_macro"],
                "f1_weighted": m["f1_weighted"],
            })

            f.write(f"accuracy: {m['accuracy']:.4f}\n")
            f.write(f"f1_macro: {m['f1_macro']:.4f}\n")
            f.write(f"f1_weighted: {m['f1_weighted']:.4f}\n")

        elif task == "regression":
            m = summary["metrics"]
            record.update({
                "MAE": m["MAE"],
                "RMSE": m["RMSE"],
                "R2": m["R2"],
                "Pearson_r": m["Pearson_r"],
            })

            f.write(f"MAE: {m['MAE']:.4f}\n")
            f.write(f"RMSE: {m['RMSE']:.4f}\n")
            f.write(f"R2: {m['R2']:.4f}\n")
            f.write(f"Pearson_r: {m['Pearson_r']:.4f}\n")

        elif task == "dual":
            cls = summary["classification_metrics"]
            reg = summary["regression_metrics"]

            record.update({
                "cls_accuracy": cls["accuracy"],
                "cls_f1_macro": cls["f1_macro"],
                "cls_f1_weighted": cls["f1_weighted"],
                "reg_MAE": reg["MAE"],
                "reg_RMSE": reg["RMSE"],
                "reg_R2": reg["R2"],
                "reg_Pearson_r": reg["Pearson_r"],
            })

            f.write(f"cls_accuracy: {cls['accuracy']:.4f}\n")
            f.write(f"cls_f1_macro: {cls['f1_macro']:.4f}\n")
            f.write(f"cls_f1_weighted: {cls['f1_weighted']:.4f}\n")
            f.write(f"reg_MAE: {reg['MAE']:.4f}\n")
            f.write(f"reg_RMSE: {reg['RMSE']:.4f}\n")
            f.write(f"reg_R2: {reg['R2']:.4f}\n")
            f.write(f"reg_Pearson_r: {reg['Pearson_r']:.4f}\n")

        all_records.append(record)
        f.write("\n")

cls_records = [r for r in all_records if r["task"] == "classification"]
reg_records = [r for r in all_records if r["task"] == "regression"]
dual_records = [r for r in all_records if r["task"] == "dual"]

print("\n===== Best performance per task =====")

# classification: highest f1_macro
if len(cls_records) > 0:
    best_cls = sorted(
        cls_records,
        key=lambda r: (-r["f1_macro"], -r["accuracy"], -r["f1_weighted"])
    )[0]

    print("\nBest Classification:")
    print(best_cls["folder"])
    print(best_cls)

# regression: lowest MAE
if len(reg_records) > 0:
    best_reg = sorted(
        reg_records,
        key=lambda r: (r["MAE"], r["RMSE"], -r["R2"], -r["Pearson_r"])
    )[0]

    print("\nBest Regression:")
    print(best_reg["folder"])
    print(best_reg)

# dual - by rank sum
if len(dual_records) > 0:
    import pandas as pd

    df = pd.DataFrame(dual_records)
    df["rank_cls"] = df["cls_f1_macro"].rank(ascending=False, method="min")
    df["rank_reg"] = df["reg_MAE"].rank(ascending=True, method="min")
    df["rank_sum"] = df["rank_cls"] + df["rank_reg"]

    df = df.sort_values(by=["rank_sum", "rank_cls", "rank_reg"], ascending=[True, True, True])

    best_dual = df.iloc[0].to_dict()

    print("\nBest Dual:")
    print(best_dual["folder"])
    print(best_dual)

print(f"\nSaved full summary to: {out_txt}")


===== Best performance per task =====

Best Classification:
abmil_classification_20260422_142528_0
{'folder': 'abmil_classification_20260422_142528_0', 'task': 'classification', 'accuracy': 0.8756756756756757, 'f1_macro': 0.7753486610134918, 'f1_weighted': 0.8882445498212611}

Best Regression:
abmil_regression_20260422_144229_1
{'folder': 'abmil_regression_20260422_144229_1', 'task': 'regression', 'MAE': 3.824697732925415, 'RMSE': 5.500482559204102, 'R2': 0.9579639434814453, 'Pearson_r': 0.9788508415222168}

Best Dual:
abmil_dual_20260422_144831_0
{'folder': 'abmil_dual_20260422_144831_0', 'task': 'dual', 'cls_accuracy': 0.8756756756756757, 'cls_f1_macro': 0.7794486500412982, 'cls_f1_weighted': 0.8898896099946363, 'reg_MAE': 3.5631704330444336, 'reg_RMSE': 4.989130973815918, 'reg_R2': 0.9654163718223572, 'reg_Pearson_r': 0.9831985235214233, 'rank_cls': 1.0, 'rank_reg': 2.0, 'rank_sum': 3.0}

Saved full summary to: /home/varnika/neurovfm/experiments/outputs/results_MLPv2_attwntion_wt_f

### Diagnostics

In [46]:
import pandas as pd
import pathlib
import re
import torch

path_openbhb = pathlib.Path(cfg.openbhb_embeddings_dir)
path_adni = pathlib.Path(cfg.adni_embeddings_dir)

subject_meta = pd.read_csv(cfg.openbhb_metadata_tsv, sep="\t")
adni_meta = pd.read_csv(cfg.adni_metadata_tsv, sep="\t")
diagnosis_labels = pd.read_csv(cfg.diagnosis_labels_tsv)

print("subject_meta shape:", subject_meta.shape)
print("adni_meta shape:", adni_meta.shape)
print("diagnosis_labels shape:", diagnosis_labels.shape)

print("\nsubject_meta columns:")
print(subject_meta.columns.tolist())

print("\nadni_meta columns:")
print(adni_meta.columns.tolist())

print("\ndiagnosis_labels columns:")
print(diagnosis_labels.columns.tolist())

subject_meta shape: (3227, 12)
adni_meta shape: (387, 74)
diagnosis_labels shape: (387, 16)

subject_meta columns:
['participant_id', 'study', 'sex', 'age', 'site', 'diagnosis', 'tiv', 'csfv', 'gmv', 'wmv', 'magnetic_field_strength', 'acquisition_setting']

adni_meta columns:
['source_file', 'age', 'Modality', 'MagneticFieldStrength', 'ImagingFrequency', 'Manufacturer', 'ManufacturersModelName', 'InstitutionName', 'DeviceSerialNumber', 'BodyPartExamined', 'PatientPosition', 'SoftwareVersions', 'MRAcquisitionType', 'SeriesDescription', 'ProtocolName', 'ScanningSequence', 'SequenceVariant', 'ScanOptions', 'SequenceName', 'ImageType', 'NonlinearGradientCorrection', 'SeriesNumber', 'AcquisitionTime', 'AcquisitionNumber', 'SliceThickness', 'SAR', 'EchoTime', 'RepetitionTime', 'SpoilingState', 'InversionTime', 'FlipAngle', 'PartialFourier', 'BaseResolution', 'ShimSetting', 'TxRefAmp', 'PhaseResolution', 'ReceiveCoilName', 'ReceiveCoilActiveElements', 'PulseSequenceDetails', 'RefLinesPE', 'Co

In [56]:
subject_meta["participant_id"] = subject_meta["participant_id"].astype(str)

adni_meta["participant_id"] = adni_meta["source_file"].astype(str).str.extract(r"(.+)\.json")
adni_meta["participant_id"] = adni_meta["participant_id"].astype(str)

diagnosis_labels["PTID"] = diagnosis_labels["PTID"].astype(str)

print("\nExample OpenBHB metadata IDs:")
print(subject_meta["participant_id"].head(10).tolist())

print("\nExample ADNI metadata IDs:")
print(adni_meta["participant_id"].head(10).tolist())

print("\nExample diagnosis PTIDs:")
print(diagnosis_labels["PTID"].head(10).tolist())


Example OpenBHB metadata IDs:
['100053248969', '100263562592', '100479214233', '100544064116', '101404752059', '101942030871', '102730278561', '103116472269', '103326586557', '104230378530']

Example ADNI metadata IDs:
['002_S_0413', '002_S_0559', '002_S_0729', '002_S_0816', '002_S_0954', '002_S_1018', '002_S_1070', '002_S_1155', '002_S_1261', '002_S_1268']

Example diagnosis PTIDs:
['002_S_0413', '002_S_0559', '002_S_0729', '002_S_0816', '002_S_0954', '002_S_1018', '002_S_1070', '002_S_1155', '002_S_1261', '002_S_1268']


In [57]:
openbhb_pattern = re.compile(r"sub-([^_]+)_encoder_embeddings\.pt")
adni_pattern = re.compile(r"(.+)_encoder_embeddings\.pt")

openbhb_ids = []
for file in path_openbhb.glob("*.pt"):
    match = openbhb_pattern.search(file.name)
    if match:
        openbhb_ids.append(match.group(1))

adni_ids = []
for file in path_adni.glob("*.pt"):
    match = adni_pattern.search(file.name)
    if match:
        adni_ids.append(match.group(1))

print("Number of OpenBHB embedding IDs:", len(openbhb_ids))
print("Number of ADNI embedding IDs:", len(adni_ids))

print("\nExample OpenBHB embedding IDs:")
print(openbhb_ids[:10])

print("\nExample ADNI embedding IDs:")
print(adni_ids[:10])

Number of OpenBHB embedding IDs: 1330
Number of ADNI embedding IDs: 387

Example OpenBHB embedding IDs:
['726862226876', '486972667341', '787417420604', '417519708424', '952477762961', '296656452301', '158561810586', '250511855843', '719305753553', '454953257900']

Example ADNI embedding IDs:
['052_S_1352', '023_S_0376', '068_S_0442', '033_S_1309', '023_S_0604', '002_S_0413', '031_S_2233', '027_S_1277', '027_S_0417', '007_S_1222']


In [58]:
openbhb_emb_set = set(openbhb_ids)
openbhb_meta_set = set(subject_meta["participant_id"].astype(str))

adni_emb_set = set(adni_ids)
adni_meta_set = set(adni_meta["participant_id"].astype(str))
diag_set = set(diagnosis_labels["PTID"].astype(str))

print("\nOpenBHB embedding ↔ metadata overlap:", len(openbhb_emb_set & openbhb_meta_set))
print("OpenBHB embedding only examples:", list(sorted(openbhb_emb_set - openbhb_meta_set))[:10])
print("OpenBHB metadata only examples:", list(sorted(openbhb_meta_set - openbhb_emb_set))[:10])

print("\nADNI embedding ↔ metadata overlap:", len(adni_emb_set & adni_meta_set))
print("ADNI embedding only examples:", list(sorted(adni_emb_set - adni_meta_set))[:10])
print("ADNI metadata only examples:", list(sorted(adni_meta_set - adni_emb_set))[:10])

print("\nADNI embedding ↔ diagnosis overlap:", len(adni_emb_set & diag_set))
print("ADNI embedding only examples:", list(sorted(adni_emb_set - diag_set))[:10])
print("Diagnosis only examples:", list(sorted(diag_set - adni_emb_set))[:10])


OpenBHB embedding ↔ metadata overlap: 1330
OpenBHB embedding only examples: []
OpenBHB metadata only examples: ['285593347262', '286377011623', '287060632619', '287389356878', '287470725724', '287536234169', '287677435094', '287694968928', '289431285739', '289461458220']

ADNI embedding ↔ metadata overlap: 387
ADNI embedding only examples: []
ADNI metadata only examples: []

ADNI embedding ↔ diagnosis overlap: 387
ADNI embedding only examples: []
Diagnosis only examples: []


In [59]:
print("\nDiagnosis value counts:")
print(diagnosis_labels["DIAGNOSIS"].value_counts(dropna=False).head(20))


Diagnosis value counts:
DIAGNOSIS
MCI         186
CN          162
Dementia     39
Name: count, dtype: int64


In [60]:

openbhb_data_list = [{"subject id": sid, "diagnosis": "CN"} for sid in openbhb_ids]
adni_data_list = []

for sid in adni_ids[:20]:
    row = diagnosis_labels[diagnosis_labels["PTID"].astype(str) == sid]
    dx = row["DIAGNOSIS"].iloc[0] if len(row) > 0 else None
    adni_data_list.append({"subject id": sid, "diagnosis": dx})

openbhb_emb_df = pd.DataFrame(openbhb_data_list)
adni_emb_df = pd.DataFrame(adni_data_list)

print("\nopenbhb_emb_df shape:", openbhb_emb_df.shape)
print("adni_emb_df shape:", adni_emb_df.shape)

print("\nopenbhb_emb_df head:")
print(openbhb_emb_df.head())

print("\nadni_emb_df head:")
print(adni_emb_df.head())


openbhb_emb_df shape: (1330, 2)
adni_emb_df shape: (20, 2)

openbhb_emb_df head:
     subject id diagnosis
0  726862226876        CN
1  486972667341        CN
2  787417420604        CN
3  417519708424        CN
4  952477762961        CN

adni_emb_df head:
   subject id diagnosis
0  052_S_1352       MCI
1  023_S_0376       MCI
2  068_S_0442       MCI
3  033_S_1309       MCI
4  023_S_0604       MCI


In [63]:
openbhb_df = pd.merge(
    openbhb_emb_df,
    subject_meta,
    left_on="subject id",
    right_on="participant_id",
    how="inner",
)

print("\nMerged OpenBHB shape:", openbhb_df.shape)
print(openbhb_df[["subject id", "diagnosis_x", "diagnosis_y", "participant_id"]].head())


Merged OpenBHB shape: (1330, 14)
     subject id diagnosis_x diagnosis_y participant_id
0  726862226876          CN     control   726862226876
1  486972667341          CN     control   486972667341
2  787417420604          CN     control   787417420604
3  417519708424          CN     control   417519708424
4  952477762961          CN     control   952477762961


In [64]:
print(adni_meta["source_file"].head(20).tolist())

['002_S_0413.json', '002_S_0559.json', '002_S_0729.json', '002_S_0816.json', '002_S_0954.json', '002_S_1018.json', '002_S_1070.json', '002_S_1155.json', '002_S_1261.json', '002_S_1268.json', '002_S_1280.json', '002_S_4213.json', '002_S_4225.json', '002_S_4229.json', '002_S_4473.json', '002_S_4654.json', '002_S_4799.json', '002_S_5178.json', '002_S_5230.json', '003_S_0908.json']


In [65]:
print(openbhb_df.columns.tolist())

['subject id', 'diagnosis_x', 'participant_id', 'study', 'sex', 'age', 'site', 'diagnosis_y', 'tiv', 'csfv', 'gmv', 'wmv', 'magnetic_field_strength', 'acquisition_setting']


### Hyperparameter tuning

In [25]:
import os
import json
import time
import traceback
import itertools
from datetime import datetime
from dataclasses import asdict

import numpy as np
import pandas as pd
import torch


def get_grid_search_spaces():
    classification_grid = {
        "hidden_dim": [128, 256],
        "batch_size": [8, 16],
        "learning_rate": [1e-3, 3e-4],
        "weight_decay": [0.0, 1e-4],
    }

    regression_grid = {
        "hidden_dim": [128, 256],
        "batch_size": [8, 16],
        "learning_rate": [1e-3, 3e-4],
        "weight_decay": [0.0, 1e-4],
    }

    dual_grid = {
        "hidden_dim": [128, 256],
        "batch_size": [8, 16],
        "learning_rate": [1e-3, 3e-4],
        "classification_loss_weight": [1.0, 2.0],
    }

    total_runs = (
        np.prod([len(v) for v in classification_grid.values()]) +
        np.prod([len(v) for v in regression_grid.values()]) +
        np.prod([len(v) for v in dual_grid.values()])
    )
    print(f"Total planned runs = {int(total_runs)}")

    return {
        "classification": classification_grid,
        "regression": regression_grid,
        "dual": dual_grid,
    }

def build_cfg(base_cfg: Config, overrides: dict) -> Config:
    cfg_dict = asdict(base_cfg)
    cfg_dict.update(overrides)
    return Config(**cfg_dict)


def iter_grid(grid_dict):
    keys = list(grid_dict.keys())
    values = [grid_dict[k] for k in keys]
    for combo in itertools.product(*values):
        yield dict(zip(keys, combo))


def safe_json(obj):
    def convert(x):
        if isinstance(x, (np.integer, np.floating)):
            return x.item()
        if isinstance(x, np.ndarray):
            return x.tolist()
        return str(x)
    return json.dumps(obj, default=convert, indent=2)


def write_log(log_path: str, text: str):
    with open(log_path, "a") as f:
        f.write(text + "\n")

def make_result_record(task: str, run_idx: int, params: dict, summary: dict, output_dir: str):
    record = {
        "task": task,
        "run_idx": run_idx,
        "params": params,
        "output_dir": output_dir,
        "training_time_seconds": summary.get("training_time_seconds", None),
    }

    if task == "classification":
        metrics = summary["metrics"]
        record.update({
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
        })

    elif task == "regression":
        metrics = summary["metrics"]
        record.update({
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
            "Pearson_r": metrics["Pearson_r"],
        })

    elif task == "dual":
        cls = summary["classification_metrics"]
        reg = summary["regression_metrics"]
        record.update({
            "cls_accuracy": cls["accuracy"],
            "cls_f1_macro": cls["f1_macro"],
            "cls_f1_weighted": cls["f1_weighted"],
            "reg_MAE": reg["MAE"],
            "reg_RMSE": reg["RMSE"],
            "reg_R2": reg["R2"],
            "reg_Pearson_r": reg["Pearson_r"],
        })

    return record


def select_best_classification(records):
    # highest macro-F1, then highest accuracy
    return sorted(
        records,
        key=lambda r: (-r["f1_macro"], -r["accuracy"], -r["f1_weighted"])
    )[0]


def select_best_regression(records):
    # lowest MAE, then lowest RMSE, then highest R2
    return sorted(
        records,
        key=lambda r: (r["MAE"], r["RMSE"], -r["R2"], -r["Pearson_r"])
    )[0]


def select_best_dual(records):
    # balance both tasks using rank sum:
    # classification wants higher macro-F1, regression wants lower MAE
    df = pd.DataFrame(records).copy()

    df["rank_cls_f1"] = df["cls_f1_macro"].rank(ascending=False, method="min")
    df["rank_reg_mae"] = df["reg_MAE"].rank(ascending=True, method="min")
    df["rank_sum"] = df["rank_cls_f1"] + df["rank_reg_mae"]

    df = df.sort_values(
        by=["rank_sum", "rank_cls_f1", "rank_reg_mae", "cls_accuracy"],
        ascending=[True, True, True, False]
    )

    best_run_idx = int(df.iloc[0]["run_idx"])
    best_record = [r for r in records if r["run_idx"] == best_run_idx][0]
    return best_record


def grid_search_all_tasks(base_cfg: Config = None):
    """
    Runs <= 50 total experiments across:
      - classification
      - regression
      - dual

    Logs:
      - one .txt summary log with every run + best params
      - one CSV per task with compact result table
    """
    if base_cfg is None:
        base_cfg = Config(
            epochs=200,
            batch_size=16,   # overridden per run
            task="classification",
        )

    set_seed(base_cfg.random_state)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    search_root = os.path.join(base_cfg.output_root, f"grid_search_{timestamp}")
    os.makedirs(search_root, exist_ok=True)

    log_path = os.path.join(search_root, "grid_search_log.txt")
    write_log(log_path, f"Grid search started: {timestamp}")
    write_log(log_path, f"Search root: {search_root}")
    write_log(log_path, "")

    grids = get_grid_search_spaces()
    write_log(log_path, "Search spaces:")
    write_log(log_path, safe_json(grids))
    write_log(log_path, "")

    print("Loading data once for all runs...")
    combined_df = load_combined_dataframe(base_cfg)
    train_df, eval_df = make_train_eval_split(combined_df, base_cfg)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    write_log(log_path, f"Device: {device}")
    write_log(log_path, f"Train size: {len(train_df)}")
    write_log(log_path, f"Eval size: {len(eval_df)}")
    write_log(log_path, "")

    all_results = {
        "classification": [],
        "regression": [],
        "dual": [],
    }

    for task in ["classification", "regression", "dual"]:
        grid = grids[task]
        param_list = list(iter_grid(grid))

        write_log(log_path, "=" * 80)
        write_log(log_path, f"Task: {task}")
        write_log(log_path, f"Planned runs: {len(param_list)}")
        write_log(log_path, "=" * 80)

        for run_idx, params in enumerate(param_list, start=1):
            overrides = {
                "task": task,
                "epochs": 200,
                "attention_type": "0",
                "input_dim": 768,
                "num_classes": 3,
                "random_state": 42,
                "test_size": 0.2,
                "run_name": f"grid_{task}_run_{run_idx:02d}",
            }
            overrides.update(params)

            if task != "dual":
                overrides.setdefault("classification_loss_weight", 1.0)
                overrides.setdefault("regression_loss_weight", 1.0)
            else:
                overrides.setdefault("regression_loss_weight", 1.0)
                overrides.setdefault("weight_decay", 1e-4)

            trial_cfg = build_cfg(base_cfg, overrides)

            print(f"[{task}] Run {run_idx}/{len(param_list)} | params = {params}")
            write_log(log_path, "-" * 80)
            write_log(log_path, f"Run {run_idx}/{len(param_list)} | task = {task}")
            write_log(log_path, f"Params: {params}")

            try:
                _, _, train_loader, eval_loader = create_dataloaders(train_df, eval_df, trial_cfg)

                output_dir = create_output_dir(trial_cfg)
                run_start = time.time()

                if task == "classification":
                    summary = run_classification(
                        trial_cfg, train_df, eval_df, train_loader, eval_loader, device, output_dir
                    )
                elif task == "regression":
                    summary = run_regression(
                        trial_cfg, train_df, eval_df, train_loader, eval_loader, device, output_dir
                    )
                elif task == "dual":
                    summary = run_dual(
                        trial_cfg, train_df, eval_df, train_loader, eval_loader, device, output_dir
                    )
                else:
                    raise ValueError(f"Unknown task: {task}")

                elapsed = time.time() - run_start
                record = make_result_record(task, run_idx, params, summary, output_dir)
                record["elapsed_seconds_wallclock"] = elapsed
                all_results[task].append(record)

                write_log(log_path, f"Status: Success")
                write_log(log_path, f"Output dir: {output_dir}")
                write_log(log_path, f"Elapsed: {elapsed:.2f} sec")

                if task == "classification":
                    write_log(
                        log_path,
                        f"Metrics | acc={record['accuracy']:.4f}, "
                        f"f1_macro={record['f1_macro']:.4f}, "
                        f"f1_weighted={record['f1_weighted']:.4f}"
                    )
                elif task == "regression":
                    write_log(
                        log_path,
                        f"Metrics | MAE={record['MAE']:.4f}, "
                        f"RMSE={record['RMSE']:.4f}, "
                        f"R2={record['R2']:.4f}, "
                        f"Pearson_r={record['Pearson_r']:.4f}"
                    )
                else:
                    write_log(
                        log_path,
                        f"Metrics | cls_f1_macro={record['cls_f1_macro']:.4f}, "
                        f"cls_acc={record['cls_accuracy']:.4f}, "
                        f"reg_MAE={record['reg_MAE']:.4f}, "
                        f"reg_R2={record['reg_R2']:.4f}"
                    )

            except Exception as e:
                write_log(log_path, f"Status: FAILED")
                write_log(log_path, f"Error: {repr(e)}")
                write_log(log_path, traceback.format_exc())
                print(f"Run failed for task={task}, run={run_idx}: {e}")

        if len(all_results[task]) > 0:
            pd.DataFrame(all_results[task]).to_csv(
                os.path.join(search_root, f"{task}_grid_results.csv"),
                index=False
            )

        write_log(log_path, "")

    write_log(log_path, "=" * 80)
    write_log(log_path, "Best hyperparameters by task:")
    write_log(log_path, "=" * 80)

    best_by_task = {}

    if len(all_results["classification"]) > 0:
        best_cls = select_best_classification(all_results["classification"])
        best_by_task["classification"] = best_cls
        write_log(log_path, "[Classification] Best run:")
        write_log(log_path, safe_json(best_cls))
        write_log(log_path, "")

    if len(all_results["regression"]) > 0:
        best_reg = select_best_regression(all_results["regression"])
        best_by_task["regression"] = best_reg
        write_log(log_path, "[Regression] Best run:")
        write_log(log_path, safe_json(best_reg))
        write_log(log_path, "")

    if len(all_results["dual"]) > 0:
        best_dual = select_best_dual(all_results["dual"])
        best_by_task["dual"] = best_dual
        write_log(log_path, "[Dual] Best run:")
        write_log(log_path, safe_json(best_dual))
        write_log(log_path, "")

    with open(os.path.join(search_root, "best_hyperparameters.json"), "w") as f:
        json.dump(best_by_task, f, indent=2, default=str)

    with open(os.path.join(search_root, "best_hyperparameters.txt"), "w") as f:
        f.write(json.dumps(best_by_task, indent=2, default=str))

    print(f"\nGrid search complete.")
    print(f"Text log saved to: {log_path}")
    print(f"Results folder: {search_root}")

    return {
        "search_root": search_root,
        "log_path": log_path,
        "all_results": all_results,
        "best_by_task": best_by_task,
    }

In [26]:
def grid_search_dual(base_cfg: Config = None):
    """
    Runs grid search only for the dual task.

    Logs:
      - one .txt summary log with every run + best params
      - one CSV with compact result table
    """
    if base_cfg is None:
        base_cfg = Config(
            epochs=200,
            batch_size=16,
            task="dual",
        )

    set_seed(base_cfg.random_state)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    search_root = os.path.join(base_cfg.output_root, f"grid_search_dual_{timestamp}")
    os.makedirs(search_root, exist_ok=True)

    log_path = os.path.join(search_root, "grid_search_log.txt")
    write_log(log_path, f"Dual-only grid search started: {timestamp}")
    write_log(log_path, f"Search root: {search_root}")
    write_log(log_path, "")

    grids = get_grid_search_spaces()
    dual_grid = grids["dual"]

    write_log(log_path, "Search space:")
    write_log(log_path, safe_json({"dual": dual_grid}))
    write_log(log_path, "")

    print("Loading data once for all runs...")
    combined_df = load_combined_dataframe(base_cfg)
    train_df, eval_df = make_train_eval_split(combined_df, base_cfg)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    write_log(log_path, f"Device: {device}")
    write_log(log_path, f"Train size: {len(train_df)}")
    write_log(log_path, f"Eval size: {len(eval_df)}")
    write_log(log_path, "")

    all_results = {"dual": []}

    task = "dual"
    param_list = list(iter_grid(dual_grid))

    write_log(log_path, "=" * 80)
    write_log(log_path, f"Task: {task}")
    write_log(log_path, f"Planned runs: {len(param_list)}")
    write_log(log_path, "=" * 80)

    for run_idx, params in enumerate(param_list, start=1):
        overrides = {
            "task": "dual",
            "epochs": 200,
            "attention_type": "0",
            "input_dim": 768,
            "num_classes": 3,
            "random_state": 42,
            "test_size": 0.2,
            "run_name": f"grid_dual_run_{run_idx:02d}",
            "regression_loss_weight": 1.0,
            "weight_decay": 1e-4,
        }
        overrides.update(params)

        trial_cfg = build_cfg(base_cfg, overrides)

        print(f"[dual] Run {run_idx}/{len(param_list)} | params = {params}")
        write_log(log_path, "-" * 80)
        write_log(log_path, f"Run {run_idx}/{len(param_list)} | task = dual")
        write_log(log_path, f"Params: {params}")

        try:
            _, _, train_loader, eval_loader = create_dataloaders(train_df, eval_df, trial_cfg)

            output_dir = create_output_dir(trial_cfg)
            run_start = time.time()

            summary = run_dual(
                trial_cfg, train_df, eval_df, train_loader, eval_loader, device, output_dir
            )

            elapsed = time.time() - run_start
            record = make_result_record("dual", run_idx, params, summary, output_dir)
            record["elapsed_seconds_wallclock"] = elapsed
            all_results["dual"].append(record)

            write_log(log_path, "Status: Success")
            write_log(log_path, f"Output dir: {output_dir}")
            write_log(log_path, f"Elapsed: {elapsed:.2f} sec")
            write_log(
                log_path,
                f"Metrics | cls_f1_macro={record['cls_f1_macro']:.4f}, "
                f"cls_acc={record['cls_accuracy']:.4f}, "
                f"reg_MAE={record['reg_MAE']:.4f}, "
                f"reg_R2={record['reg_R2']:.4f}"
            )

        except Exception as e:
            write_log(log_path, "Status: FAILED")
            write_log(log_path, f"Error: {repr(e)}")
            write_log(log_path, traceback.format_exc())
            print(f"Run failed for task=dual, run={run_idx}: {e}")

    if len(all_results["dual"]) > 0:
        pd.DataFrame(all_results["dual"]).to_csv(
            os.path.join(search_root, "dual_grid_results.csv"),
            index=False
        )

    write_log(log_path, "")
    write_log(log_path, "=" * 80)
    write_log(log_path, "Best hyperparameters:")
    write_log(log_path, "=" * 80)

    best_by_task = {}

    if len(all_results["dual"]) > 0:
        best_dual = select_best_dual(all_results["dual"])
        best_by_task["dual"] = best_dual
        write_log(log_path, "[Dual] Best run:")
        write_log(log_path, safe_json(best_dual))
        write_log(log_path, "")

    with open(os.path.join(search_root, "best_hyperparameters.json"), "w") as f:
        json.dump(best_by_task, f, indent=2, default=str)

    with open(os.path.join(search_root, "best_hyperparameters.txt"), "w") as f:
        f.write(json.dumps(best_by_task, indent=2, default=str))

    print("\nDual-only grid search complete.")
    print(f"Text log saved to: {log_path}")
    print(f"Results folder: {search_root}")

    return {
        "search_root": search_root,
        "log_path": log_path,
        "all_results": all_results,
        "best_by_task": best_by_task,
    }

In [24]:
base_cfg = Config(
    epochs=200,
    batch_size=16, # overwritten by grid
    input_dim=768,
    hidden_dim=256, # overwritten by grid
    num_classes=3,
    learning_rate=1e-3, # overwritten by grid
    weight_decay=1e-4, # overwritten for cls/reg, fixed for dual unless you change it
    attention_type="0",
    early_stopping = True,
    patience = 10,
    min_delta = 1e-4,
    classification_loss_weight=1.0, # overwritten for cls/reg
    regression_loss_weight=1.0, # overwritten for cls/reg
)

grid_search_all_tasks(base_cfg)
search_results = grid_search_all_tasks(base_cfg)

Total planned runs = 48
Loading data once for all runs...


/tmp/ipykernel_2105804/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64
[classification] Run 1/16 | params = {'hidden_dim': 128, 'batch_size': 8, 'learning_rate': 0.001, 'weight_decay': 0.0}
Epoch [10/200] | Train Loss: 0.0989, Acc: 95.81% | Val Loss: 0.6055, Acc: 85.95%
Early stopping triggered at epoch 14
Restore

/tmp/ipykernel_2105804/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64
[classification] Run 1/16 | params = {'hidden_dim': 128, 'batch_size': 8, 'learning_rate': 0.001, 'weight_decay': 0.0}
Epoch [10/200] | Train Loss: 0.0989, Acc: 95.81% | Val Loss: 0.6055, Acc: 85.95%
Early stopping triggered at epoch 14
Restore

KeyboardInterrupt: 

In [29]:
base_cfg = Config(
    epochs=200,
    batch_size=16, # overwritten by grid
    input_dim=768,
    hidden_dim=256, # overwritten by grid
    num_classes=3,
    learning_rate=1e-3, # overwritten by grid
    weight_decay=1e-4, # overwritten for cls/reg, fixed for dual unless you change it
    attention_type="0",
    early_stopping = True,
    patience = 10,
    min_delta = 1e-4,
    classification_loss_weight=1.0, # overwritten for cls/reg
    regression_loss_weight=1.0, # overwritten for cls/reg
)

search_results = grid_search_dual(base_cfg)

Total planned runs = 48
Loading data once for all runs...


/tmp/ipykernel_4156593/3821270671.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location="cpu")



Dataframe sizes before metadata merge:
OpenBHB embeddings: (1330, 4)
ADNI embeddings: (520, 3)

Dataframe sizes after metadata merge:
OpenBHB merged: (1330, 16)
ADNI merged: (520, 7)

Diagnosis distribution before filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Diagnosis distribution after filtering:
diagnosis
CN          1492
Dementia     209
MCI          149
Name: count, dtype: int64

Final combined dataframe shape: (1850, 19)

Source distribution:
source
OpenBHB    1330
ADNI        520
Name: count, dtype: int64

Source × diagnosis breakdown:
source   diagnosis
ADNI     Dementia      209
         CN            162
         MCI           149
OpenBHB  CN           1330
Name: count, dtype: int64
[dual] Run 1/16 | params = {'hidden_dim': 128, 'batch_size': 8, 'learning_rate': 0.001, 'classification_loss_weight': 1.0}
Epoch [10/200] | Train Total: 23.5248 | Val Total: 30.0582 | Train Acc: 90.34% | Val Acc: 87.57% | Train MAE: 3.6088 | Va